# Foodie Advisor · 2. El detector de fallas del ciclo de servicio

Este notebook reúne las fases `03_lexico`, `04_anotacion`, `05_clasificador`, `05b_calibracion_umbral`, `06_evaluacion`. Cada sección escribe su `resultados/hallazgos_XX.json`, que es lo que lee la app.

## 03 · Léxico de etiquetas débiles: del regex del avance al v2

| | |
|---|---|
| **Qué entra** | oraciones y reseñas de Madrid, solo restaurantes de train (del 00) |
| **Qué sale** | `lexico.ETAPAS_V2` en el paquete, que usa el 05 para las etiquetas débiles |

El clasificador del 05 aprende de etiquetas débiles: una oración es positiva para una etapa si la marca el regex. Si el regex es ruidoso, el modelo aprende ese ruido. Aquí medimos cada término del regex del avance (v1), buscamos candidatos nuevos con POS + PMI y armamos un v2 con una regla explícita de qué entra.

El v1 **no se modifica**: sigue siendo el escalón "regex actual" de la escalera y define el subconjunto de paráfrasis del gold. Todo se mide en los restaurantes de train de Madrid; Barcelona y el dev no se miran.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from foodie_advisor import datos, lexico, etiquetas as et

pd.set_option("display.max_colwidth", 140)
orac = datos.cargar_oraciones("Madrid", con_resena=True)
rest_tr, _ = et.particion_restaurantes(orac.restaurante)
o = orac[orac.restaurante.isin(rest_tr)].reset_index(drop=True)
r = datos.cargar_resenas("Madrid", ["review_id", "restaurante", "estrellas", "texto"])
r = r[r.restaurante.isin(rest_tr)].reset_index(drop=True)
minus = o.oracion.str.lower()
NEG, CINCO = (o.estrellas <= 2).values, (o.estrellas == 5).values
print(f"Madrid train: {len(rest_tr)} restaurantes, {len(r):,} reseñas, {len(o):,} oraciones")

Madrid train: 1201 restaurantes, 128,122 reseñas, 634,033 oraciones


### 1. ¿Cuánto ve el regex del avance?

La cobertura se puede medir de dos maneras, y dan cifras muy distintas. **Por reseña**: qué parte de las reseñas de 1-2★ tiene al menos una falla marcada. **Por oración**: qué parte de las oraciones de esas reseñas está marcada. El umbral original (más del 54.5%) es por reseña.

In [2]:
def cobertura(etapas, guardia=False):
    Yo = et.etiquetas_regex(o.oracion[NEG], etapas, guardia)
    Yr = et.etiquetas_regex(r.texto[r.estrellas <= 2], etapas, guardia)
    return {"por reseña, 7 etapas %": Yr.any(axis=1).mean() * 100, "por reseña, ciclo %": Yr[et.CICLO].any(axis=1).mean() * 100,
            "por oración, 7 etapas %": Yo.any(axis=1).mean() * 100, "por oración, ciclo %": Yo[et.CICLO].any(axis=1).mean() * 100}

cob = pd.DataFrame({"v1 (avance)": cobertura(lexico.ETAPAS)}).T.round(1)
cob

,"por reseña, 7 etapas %","por reseña, ciclo %","por oración, 7 etapas %","por oración, ciclo %"
v1 (avance),49.8,27.7,10.4,5.2


### 2. Término por término: ¿cuánto separa una reseña mala de una buena?

Si un término marca de verdad una falla, debería ser mucho más frecuente en oraciones de reseñas de 1-2★ que de 5★. La razón entre las dos tasas sirve como indicador de su precisión **sin anotar nada**. No reemplaza al gold, pero ordena los términos.

In [3]:
def medir(rx, guardia=False):
    m = minus.str.contains(rx).values
    if guardia and m.any():
        idx = np.flatnonzero(m)
        m[idx] = [any(not lexico.negado(x, k.start()) for k in __import__("re").finditer(rx, x)) for x in minus.values[idx]]
    a, b = m[NEG].mean() * 100, m[CINCO].mean() * 100
    return {"oraciones": int(m.sum()), "% en 1-2★": round(a, 3), "% en 5★": round(b, 4),
            "razón 1-2★ / 5★": round(a / b, 1) if b > 0 else np.inf}

info = lexico.info()
v1 = pd.DataFrame({t: medir(x.rx) for t, x in info[info.tipo == "falla"].iterrows()}).T
v1.insert(0, "etapa", info.etapa)
v1.sort_values("razón 1-2★ / 5★")

,etapa,oraciones,% en 1-2★,% en 5★,razón 1-2★ / 5★
ruido,Fuera del ciclo: local,2002.0,0.292,0.2193,1.3
espera de mesa / reserva,1. Llegada y mesa,1177.0,0.271,0.1556,1.7
caro,Fuera del ciclo: precio,7803.0,2.582,0.6186,4.2
olvidos,2. Atención en sala,739.0,0.361,0.0616,5.9
lentitud,3. Cocina → mesa,2242.0,0.818,0.1373,6.0
salado,Fuera del ciclo: cocina,1049.0,0.462,0.0610,7.6
trampa turística,Fuera del ciclo: precio,670.0,0.391,0.0469,8.3
grasoso,Fuera del ciclo: cocina,719.0,0.405,0.0416,9.8
error en la cuenta,4. Cobro,59.0,0.036,0.0021,17.4
cobro indebido,4. Cobro,646.0,0.432,0.0236,18.3


Los términos del v1 no son igual de confiables. "Trato grosero" (89×), "pedido equivocado" (86×), "sin sabor" (65×) e "ignorados" (65×) casi solo aparecen en reseñas malas. En el otro extremo están "ruido" (1.3×), "espera de mesa / reserva" (1.7×) y "caro" (4.2×), que aparecen casi igual en reseñas de 5★. Tres oraciones reales de reseñas de 5★ en Madrid train: *"It is loud and busy, but it's a wonderful experience."*, *"this place is busy but well worth a short wait for a table"* y *"It is quite expensive, but not at all over priced."* Para esas etapas, el regex es un detector de **tema** más que de **falla**. Por eso Llegada es la etapa más difícil de etiquetar débilmente, y por eso se sobre-muestreó en el gold.

### 3. Candidatos nuevos con POS + PMI

Etiquetamos con `nltk.pos_tag` 30,000 oraciones de reseñas de 1-2★ **que el regex no marca** y 30,000 de reseñas de 5★. Con `RegexpParser` extraemos tres patrones de frase: adjetivo + sustantivo (AN), verbo + sustantivo (VN) y sustantivo + verbo + adjetivo (NV). Cada frase se ordena por su PMI con la clase 1-2★: con clases del mismo tamaño, PMI = 1 significa que la frase solo aparece en reseñas malas.

In [4]:
rng = np.random.default_rng(0)
sin_regex = ~et.etiquetas_regex(o.oracion).any(axis=1).values
A = o.oracion[NEG & sin_regex].sample(30_000, random_state=0)
B = o.oracion[CINCO].sample(30_000, random_state=0)
cand = lexico.pmi_clase(lexico.frases_pos(A), lexico.frases_pos(B))
print(f"Frases con 12+ apariciones en 1-2★: {len(cand):,}")
for p in ["NV", "VN", "AN"]:
    display(cand[cand["patrón"] == p].head(15))

Frases con 12+ apariciones en 1-2★: 216


,patrón,frase,n en 1-2★,n en 5★,PMI
3,NV,service was terrible,28,0,1.000000
8,NV,food was average,23,0,1.000000
9,NV,service was awful,19,0,1.000000
17,NV,food was terrible,15,0,1.000000
25,NV,service was poor,13,0,1.000000
49,NV,food is ok,22,1,0.935870
53,NV,food was ok,41,2,0.931287
58,NV,service was ok,17,1,0.917538
59,NV,food is average,14,1,0.900464
97,NV,food is good,29,8,0.648528


,patrón,frase,n en 1-2★,n en 5★,PMI
1,VN,avoid this place,40,0,1.000000
7,VN,waste your money,24,0,1.000000
10,VN,was mediocre,19,0,1.000000
11,VN,save your money,19,0,1.000000
21,VN,take advantage,14,0,1.000000
24,VN,was a disappointment,13,0,1.000000
33,VN,had nothing,12,0,1.000000
35,VN,told the waitress,12,0,1.000000
37,VN,avoid this restaurant,12,0,1.000000
40,VN,spend your money,12,0,1.000000


,patrón,frase,n en 1-2★,n en 5★,PMI
0,AN,poor service,42,0,1.0
2,AN,terrible service,31,0,1.0
4,AN,worst i,26,0,1.0
5,AN,poor quality,25,0,1.0
6,AN,only good thing,25,0,1.0
12,AN,worst meal,18,0,1.0
13,AN,worst experience,17,0,1.0
14,AN,worst service,17,0,1.0
15,AN,terrible experience,17,0,1.0
16,AN,other place,16,0,1.0


Las frases con PMI alto son de tres tipos:

- **Fallas del ciclo con palabras que el regex no tenía:** "poor service", "terrible service", "service was awful", "take our order" (esperar a que alguien lo tome), "asked for the bill".
- **Fallas fuera del ciclo:** "food was terrible", "poor quality", "waste your money".
- **Juicios generales sin etapa:** "avoid this place", "worst experience", "was a joke". Estos no entran: dicen que la experiencia fue mala, pero no **qué** falló, y el gerente no puede actuar sobre eso.

También aparecen frases de puro contexto ("spanish couple", "french fries") que tienen PMI alto solo por azar con frecuencias bajas. El PMI propone y una lectura cualitativa y sistemática de las frases decide su incorporación.

### 4. De los candidatos a regex: la regla para entrar al v2

Los candidatos se agruparon en familias por etapa tras una revisión exhaustiva de las frases (por ejemplo, "terrible service", "service was awful" y "bad service" se consolidan en una sola regex), evaluando cada familia igual que los términos del v1. **Regla:** entra al v2 si aparece en 50 oraciones o más y es al menos 10 veces más frecuente en 1-2★ que en 5★. Los umbrales se fijaron antes de mirar los resultados, tomando como referencia los términos del v1 que mejor separan.

In [5]:
filas = {}
for e, d in lexico.ETAPAS_V2.items():
    for nombre, (tipo, rx) in d.items():
        if nombre.endswith("(v2)"):
            filas[nombre] = {"etapa": e, **medir(rx)}
for nombre, rx in lexico.DESCARTADOS_V2.items():
    filas[nombre] = {"etapa": "—", **medir(rx)}
nuevos = pd.DataFrame(filas).T
nuevos["entra"] = (nuevos.oraciones >= 50) & (nuevos["razón 1-2★ / 5★"] >= 10)
nuevos.sort_values("razón 1-2★ / 5★", ascending=False)

,etapa,oraciones,% en 1-2★,% en 5★,razón 1-2★ / 5★,entra
nadie atendió (v2),2. Atención en sala,86,0.08,0.0,inf,True
servicio malo (v2),2. Atención en sala,1668,1.666,0.015,110.8,True
maleducado (v2),2. Atención en sala,531,0.527,0.0059,89.5,True
comida mala (v2),Fuera del ciclo: cocina,1512,1.535,0.0248,62.0,True
tirar el dinero (v2),Fuera del ciclo: precio,825,0.691,0.0147,46.9,True
agregado a la cuenta (v2),4. Cobro,103,0.082,0.0021,39.9,True
la comida tardó X minutos,—,32,0.028,0.0012,23.7,False
apurados,—,148,0.058,0.01,5.8,False
rechazo en la puerta,—,164,0.05,0.0239,2.1,False


In [6]:
def ejemplos(rx, n_neg=3, n_5=2):
    m = minus.str.contains(rx).values
    for x in o.oracion[m & NEG].sample(min(n_neg, (m & NEG).sum()), random_state=0): print("   1-2★ |", x[:150])
    for x in o.oracion[m & CINCO].sample(min(n_5, (m & CINCO).sum()), random_state=0): print("     5★ |", x[:150])

for nombre in ["servicio malo (v2)", "agregado a la cuenta (v2)"]:
    print(nombre); ejemplos(info_v2 := lexico.info(lexico.ETAPAS_V2).rx[nombre]); print()
for nombre in ["apurados", "rechazo en la puerta"]:
    print(nombre, "(descartado)"); ejemplos(lexico.DESCARTADOS_V2[nombre]); print()

servicio malo (v2)


   1-2★ | However, our party of 11 were seated in a damp and smelly basement and treated to terrible service.
   1-2★ | Good food,too bad the service was awful.We had to ask the waiter twice to clean thw sticky table and he did it while we were holding our plates.Also,t
   1-2★ | Its a shame - I hate it when places assume that just because you are a visitor then its ok to give bad service.
     5★ | Don't mind what you read about bad service, it doesn't.
     5★ | Not only the food in excellent, consistent and uniform, the service is so but so good that separate Ten con Ten from the other ones, including the oth

agregado a la cuenta (v2)


   1-2★ | Yet all items appeared on the bill.
   1-2★ | No matter, it appeared on the bill.
   1-2★ | All of the dishes were barely warm or cold and when the erroneous dish was still added to our bill we pointed this out too, much to the chagrin of the
     5★ | The dishes were a little pricier than what I'd normally pay when travelling but the extras given to us and not charged more than made up for it!
     5★ | also we noticed that one glass of wine was not included in the bill and when we notified the waiter he told us not to worry, its on them...

apurados (descartado)


   1-2★ | The service was very poor, we felt rushed, the meals were mediocre (at best) and there are many other affordable and fabulous meals to be had in Madri
   1-2★ | The first time I ever felt rushed at a restaurant in Spain.
   1-2★ | We felt rushed while eating.
     5★ | Having a toddler we couldn't eat too fast but we never felt rushed.
     5★ | Never felt rushed, sat there and watched the world go by for as long as I wanted.

rechazo en la puerta (descartado)


   1-2★ | I actually went on their website a few days before and booked a priority reservation so that we wouldn't have a long wait when we arrived and this was
   1-2★ | There were roughly 25-30 people standing in line in back of us (who had paid for tickets online) who were all turned away.
   1-2★ | Wife and I came here for our last night in Madrid after being turned away the previous evening due it it being fully booked.
     5★ | We saw many people turned away from this tiny place.
     5★ | We saw several diners turned away, due to no space.



Seis familias pasan la regla. Las más fuertes son "servicio malo" (1,668 oraciones, 111×) y "maleducado" (531, 90×). "Nadie atendió" no aparece nunca en reseñas de 5★ dentro de Madrid train. Tres quedan fuera, por razones que se ven en los ejemplos:

- **apurados (5.8×):** en reseñas de 5★ abunda "we never felt rushed".
- **rechazo en la puerta (2.1×):** "we saw many people turned away" es señal de éxito, no de falla.
- **la comida tardó X minutos:** buena razón (24×), pero solo 32 oraciones.

De la Llegada, ningún candidato sobrevive. Sigue siendo la etapa con peores etiquetas débiles, y el modelo tendrá que aprenderla sobre todo de las oraciones anotadas.

### 5. Negación y un falso positivo frecuente

La lección del 02 vale aquí también: "not slow" o "never rude" contienen el término pero dicen lo contrario. El v2 descarta una coincidencia si hay un negador en las 3 palabras anteriores. Además, "slow" marcaba platos como "slow cooked oxtail", que abundan en reseñas de 5★.

In [7]:
guardia = pd.DataFrame({t: {"razón v1": v1.loc[t, "razón 1-2★ / 5★"],
                             "razón con guardia": medir(x.rx, guardia=True)["razón 1-2★ / 5★"]}
                         for t, x in info[info.tipo == "falla"].iterrows()})
guardia = guardia.T
guardia["oraciones con guardia"] = [medir(x.rx, guardia=True)["oraciones"] for _, x in info[info.tipo == "falla"].iterrows()]
display(guardia.sort_values("razón v1"))

lento_v1, lento_v2 = info.rx["lentitud"], lexico.info(lexico.ETAPAS_V2).rx["lentitud"]
print("lentitud v1:", medir(lento_v1), "\nlentitud v2:", medir(lento_v2))
print("Ejemplos en 5★ que el v2 ya no marca:")
for x in o.oracion[minus.str.contains(lento_v1).values & ~minus.str.contains(lento_v2).values & CINCO][:4]: print("  ", x[:120])

,razón v1,razón con guardia,oraciones con guardia
ruido,1.3,1.6,1762
espera de mesa / reserva,1.7,1.8,1099
caro,4.2,6.1,6427
olvidos,5.9,6.2,711
lentitud,6.0,6.0,2189
salado,7.6,8.5,986
trampa turística,8.3,12.8,555
grasoso,9.8,17.1,605
error en la cuenta,17.4,16.7,57
cobro indebido,18.3,24.3,561


lentitud v1: {'oraciones': 2242, '% en 1-2★': 0.818, '% en 5★': 0.1373, 'razón 1-2★ / 5★': 6.0} 
lentitud v2: {'oraciones': 2055, '% en 1-2★': 0.808, '% en 5★': 0.099, 'razón 1-2★ / 5★': 8.2}
Ejemplos en 5★ que el v2 ya no marca:


   Our main courses were the Slow Cooked Lamb with Mint Chimichurri and the Pork Chumarro with Confit Onions.
   and the oxtails speak of long slow braising with some wine and the simplest of seasonings the meals come with a few welc
   Follow the advice of the host in meals selection, don't miss slow cooked ox tail, and chocolate tart for dessert.
   one main dish was a slow cooked veal cheek very rich in flavour and the other was a steak, superb cut served as cut piec


La guardia de negación sube la razón en casi todos los términos, sin perder casi nada. "Cobro indebido" pasa de 18× a 24× (en 5★ aparecen frases como *"Queso and Guac no extra charge."*), "grasoso" de 10× a 17× (*"so light and crispy and not greasy"*), "caro" de 4.2× a 6.1× y "trato grosero" de 89× a 105×. Quitar "slow cooked" sube "lentitud" de 6.0× a 8.2×. Nada de esto toca a "ruido" ni a "espera de mesa": su problema no es la negación sino que el tema aparece en experiencias buenas.

### 6. v1 contra v2

In [8]:
cob.loc["v2 (+6 términos, sin 'slow cooked', con guardia)"] = pd.Series(cobertura(lexico.ETAPAS_V2, guardia=True)).round(1)
display(cob)

Y1 = et.etiquetas_regex(o.oracion, lexico.ETAPAS)
Y2 = et.etiquetas_regex(o.oracion, lexico.ETAPAS_V2, guardia_negacion=True)
pos = pd.DataFrame({"positivos v1": Y1.sum(), "positivos v2": Y2.sum(),
                    "razón 1-2★/5★ v1": [round(Y1[e][NEG].mean() / Y1[e][CINCO].mean(), 1) for e in et.ETIQUETAS],
                    "razón 1-2★/5★ v2": [round(Y2[e][NEG].mean() / Y2[e][CINCO].mean(), 1) for e in et.ETIQUETAS]})
pos

,"por reseña, 7 etapas %","por reseña, ciclo %","por oración, 7 etapas %","por oración, ciclo %"
v1 (avance),49.8,27.7,10.4,5.2
"v2 (+6 términos, sin 'slow cooked', con guardia)",59.2,34.9,13.4,7.0


,positivos v1,positivos v2,razón 1-2★/5★ v1,razón 1-2★/5★ v2
llegada,1177,1099,1.7,1.8
atencion,3722,5485,32.1,48.0
cocina_mesa,3143,2905,9.4,12.6
cobro,700,702,18.1,25.7
sabor,3130,4302,16.4,30.4
precio,8419,7637,4.4,7.6
local,2410,2151,2.9,3.5


El v2 ve más y con menos ruido. La cobertura por reseña de 1-2★ sube de 49.8% a 59.2% en las 7 etapas, y de 27.7% a 34.9% en el ciclo. Por oración, sube de 10.4% a 13.4%. Los positivos de Atención aumentan de 3,722 a 5,485, y su razón 1-2★/5★ sube de 32 a 48. En todas las etapas la razón sube o se mantiene, y Llegada sigue en 1.8.

**Ojo con el umbral original de cobertura (más del 54.5% de las reseñas de 1-2★):** el regex v2 ya lo supera, porque cuenta las 7 etapas incluyendo términos ruidosos como "caro" y "ruido". La cobertura sin precisión no dice nada. En el 06, la cobertura del modelo se reporta junto con su precisión en el gold, y lo que decide es el F1 y la mejora en el subconjunto de paráfrasis.

### Cifras clave


In [9]:
from foodie_advisor import resultados
# Ejemplos: oraciones reales de 1-2★ que el v1 no ve y una familia nueva del v2 (ciclo) sí
fam_ciclo = [rx for e, d in lexico.ETAPAS_V2.items() if e[0].isdigit() for nom, (tipo, rx) in d.items() if nom.endswith("(v2)")]
no_v1 = ~et.etiquetas_regex(o.oracion).any(axis=1).values
largo_ok = o.oracion.str.split().str.len().between(6, 20).values
cand_ej = o.oracion[NEG & no_v1 & largo_ok & minus.str.contains("|".join(fam_ciclo)).values]
ejemplos_v1_no_ve = cand_ej.sample(3, random_state=0).tolist()
print("Ejemplos:", ejemplos_v1_no_ve)
resultados.guardar("03", {"ejemplos_v1_no_ve": ejemplos_v1_no_ve, 
    "cobertura": cob, "terminos_v1": v1[["oraciones", "razón 1-2★ / 5★"]], "n_candidatos_pos": len(cand),
    "familias_v2": nuevos[["oraciones", "razón 1-2★ / 5★", "entra"]], "guardia": guardia, "positivos_v1_v2": pos,
})

Ejemplos: ['Poor service, no one behind bar to serve!', 'At first we asked for glass of water and the waitress was so impolite...', "Nobody apologized at all for the bad service, they didn't seem to care about us, what a shame."]
hallazgos_03.json: 7 claves → cobertura, ejemplos_v1_no_ve, familias_v2, guardia, n_candidatos_pos, positivos_v1_v2, terminos_v1


PosixPath('/Users/georgealbadr/Github/Natural_Language_Processing/resultados/hallazgos_03.json')

### Hallazgos

- **Cobertura del regex del avance (v1) en reseñas de 1-2★ de Madrid train:** 49.8% por reseña y 10.4% por oración (27.7% y 5.2% si se cuenta solo el ciclo). Son dos medidas distintas, y el plan usa la de reseña.
- **El v1 mezcla términos muy confiables** (trato grosero 89×, ignorados 65×) **con detectores de tema** (ruido 1.3×, espera de mesa 1.7×, caro 4.2×). Llegada es la etapa con peores etiquetas débiles.
- **POS + PMI propone 216 frases y 6 familias pasan la regla** (≥ 50 oraciones y ≥ 10× más frecuentes en 1-2★ que en 5★): servicio malo, nadie atendió, maleducado, comida mala, tirar el dinero y agregado a la cuenta. Se descartan apurados (5.8×, por "never felt rushed"), rechazo en la puerta (2.1×) y "tardó X minutos" (32 oraciones).
- **La guardia de negación limpia las etiquetas:** cobro indebido pasa de 18× a 24× y grasoso de 10× a 17×. Quitar "slow cooked" sube lentitud de 6.0× a 8.2×.
- **El v2 cubre el 59.2% de las reseñas de 1-2★,** por encima del umbral del 54.5% sin ningún modelo. Ese umbral, solo, ya no discrimina: en el 06 la cobertura se reporta junto con la precisión.

### Verificación de citas

Toda oración citada entre comillas en este notebook tiene que existir literalmente en el corpus.

In [10]:
citas = ["It is loud and busy, but it's a wonderful experience.", 'It is quite expensive, but not at all over priced.', 'Queso and Guac no extra charge.', 'so light and crispy and not greasy', 'this place is busy but well worth a short wait for a table']
verificadas = datos.verificar_citas(citas, ('Madrid',))
print(f"{len(verificadas)} citas verificadas:", verificadas)

5 citas verificadas: {"It is loud and busy, but it's a wonderful experience.": 'Madrid', 'It is quite expensive, but not at all over priced.': 'Madrid', 'Queso and Guac no extra charge.': 'Madrid', 'so light and crispy and not greasy': 'Madrid', 'this place is busy but well worth a short wait for a table': 'Madrid'}


## 04 · Anotación: muestras, guía y acuerdo entre anotadores

| | |
|---|---|
| **Qué entra** | `data/procesado/Madrid_*.parquet`, `Barcelona_*.parquet` (del 00) |
| **Qué sale** | `data/anotacion/madrid_train.csv`, `bcn_gold.csv`, `bcn_gold_anotador2.csv`, `clave/muestreo.csv`; al volver, κ de Cohen por etapa |

El regex del avance solo ve quejas explícitas. Para saber cuánto se le escapa, y para medir un modelo que intente verlo, hacen falta oraciones etiquetadas de forma independiente del regex. **El gold fue anotado siguiendo la guía por anotadores independientes, con doble anotación en 100 oraciones para evaluar el acuerdo inter-anotador (κ de Cohen). Adicionalmente se anotaron 300 oraciones de entrenamiento de Madrid (`madrid_train.csv`).** Los anotadores solo ven la guía y su CSV. Este notebook genera dos muestras con papeles distintos:

- **`madrid_train`** (300): oraciones de reseñas de 1-2★ que el regex **no** cubre, de restaurantes de train. Le enseñan al modelo las paráfrasis y entran al entrenamiento con peso ×5.
- **`bcn_gold`** (300): Barcelona, una ciudad que el modelo nunca ve, muestreada **al azar por estratos** y no por coincidencia con el regex. Es la prueba final y se usa **una sola vez** (notebook 06).

La guía de anotación está en `data/anotacion/guia_anotacion.md`. **Este notebook nunca sobrescribe un CSV que ya existe**: si lo encuentra, comprueba que el muestreo da los mismos ids y sigue.

In [1]:
from sklearn.metrics import cohen_kappa_score
from foodie_advisor import ANOTACION, datos, etiquetas as et

CLAVE = ANOTACION / "clave"
CLAVE.mkdir(parents=True, exist_ok=True)
assert (ANOTACION / "guia_anotacion.md").exists(), "Falta la guía de anotación"
pd.set_option("display.max_colwidth", 120)

### 1. Madrid: 300 oraciones que el regex no cubre

La partición por restaurante (85% train / 15% dev, semilla 0) es la misma que usa el clasificador en el 05. Solo se muestrea de train, para que el dev siga siendo independiente.

In [2]:
madrid = datos.cargar_oraciones("Madrid", con_resena=True)
rest_tr, rest_dev = et.particion_restaurantes(madrid.restaurante)
madrid_tr = madrid[madrid.restaurante.isin(rest_tr)]

neg = madrid_tr[madrid_tr.estrellas <= 2]
cubre = et.etiquetas_regex(neg.oracion).any(axis=1).mean()
print(f"Restaurantes: {len(rest_tr)} train, {len(rest_dev)} dev")
print(f"Oraciones de reseñas 1-2★ en train: {len(neg):,} | el regex de fallas marca {cubre:.1%} de ellas")

clave_mad = et.muestra_madrid_train(madrid_tr)
clave_mad.head()

Restaurantes: 1201 train, 213 dev
Oraciones de reseñas 1-2★ en train: 75,265 | el regex de fallas marca 10.4% de ellas


,idx,grupo,estrato,N_estrato,peso
0,659598,1-2★,sin regex,67445,224.816667
1,351508,1-2★,sin regex,67445,224.816667
2,553580,1-2★,sin regex,67445,224.816667
3,679638,1-2★,sin regex,67445,224.816667
4,565765,1-2★,sin regex,67445,224.816667


### 2. Barcelona: 300 oraciones de prueba, estratificadas

Muestreo al azar en 6 estratos disjuntos: grupo de estrellas (≤3★ y 4-5★) × tema. Cobro y Llegada son raros, y con un muestreo puramente al azar el gold tendría 2 o 3 casos de cada uno. Para sobre-muestrearlos sin usar el regex de fallas se usan palabras de **tema** que no dicen si algo salió mal ("bill", "pay", "table", "reservation"). Cada oración carga su peso `N_estrato / n_estrato`; con esos pesos, el 06 reporta métricas que valen para Barcelona entera y no solo para la muestra.

In [3]:
bcn = datos.cargar_oraciones("Barcelona", con_resena=True)
clave_bcn = et.muestra_gold(bcn)
resumen = clave_bcn.groupby(["grupo", "estrato"]).agg(n=("idx", "size"), N_estrato=("N_estrato", "first"),
                                                       peso=("peso", "first"))
resumen["% de Barcelona"] = (resumen.N_estrato / len(bcn) * 100).round(2)
print(f"Barcelona: {len(bcn):,} oraciones de {bcn.restaurante.nunique():,} restaurantes")
resumen

Barcelona: 1,981,854 oraciones de 2,430 restaurantes


n  N_estrato          peso  % de Barcelona
grupo estrato                                                   
4-5★  resto          70    1332027  19028.957143           67.21
      tema cobro     15      50562   3370.800000            2.55
      tema llegada   15     155109  10340.600000            7.83
≤3★   resto         120     345763   2881.358333           17.45
      tema cobro     40      36206    905.150000            1.83
      tema llegada   40      62187   1554.675000            3.14

In [4]:
# 100 de las 300 las anota también un segundo anotador independiente, para medir el acuerdo (κ de Cohen)
rng = np.random.default_rng(1)
doble = set(rng.choice(clave_bcn.idx, 100, replace=False))
print("Subconjunto doble por estrato:", clave_bcn[clave_bcn.idx.isin(doble)].groupby("estrato").size().to_dict())

Subconjunto doble por estrato: {'resto': 66, 'tema cobro': 16, 'tema llegada': 18}


#### 2.1 Ampliación: +100 oraciones de Cobro y Llegada

Con 300 oraciones al azar, Cobro y Llegada tendrían pocos positivos y su F1 sería muy ruidoso. Por decisión de George se agregan 100 oraciones **solo** de los estratos de tema (35 + 35 en ≤3★ y 15 + 15 en 4-5★), sin repetir oraciones **ni restaurantes** del gold base. Van en un archivo aparte, `bcn_gold_extra.csv`, y `bcn_gold.csv` no se toca.

En los estratos ampliados el peso pasa a ser `N_estrato / (n_base + n_extra)`. Es una aproximación: el extra excluye los restaurantes del gold base, así que su marco es un poco menor que el del estrato completo. Los 300 restaurantes que se excluyen son pocos frente a los 2,430 de Barcelona.

In [5]:
clave_extra = et.muestra_gold_extra(bcn, clave_bcn)
assert not set(clave_extra.idx) & set(clave_bcn.idx), "oraciones repetidas"
assert not set(bcn.restaurante.loc[clave_extra.idx]) & set(bcn.restaurante.loc[clave_bcn.idx]), "restaurantes repetidos"

gold = et.combinar_pesos(clave_bcn.drop(columns="peso").assign(archivo="bcn_gold.csv"),
                         clave_extra.assign(archivo="bcn_gold_extra.csv"))
gold.groupby(["grupo", "estrato"]).agg(n_base=("archivo", lambda a: (a == "bcn_gold.csv").sum()),
                                       n_extra=("archivo", lambda a: (a == "bcn_gold_extra.csv").sum()),
                                       N_estrato=("N_estrato", "first"), peso=("peso", "first"))

n_base  n_extra  N_estrato          peso
grupo estrato                                               
4-5★  resto             70        0    1332027  19028.957143
      tema cobro        15       15      50562   1685.400000
      tema llegada      15       15     155109   5170.300000
≤3★   resto            120        0     345763   2881.358333
      tema cobro        40       35      36206    482.746667
      tema llegada      40       35      62187    829.160000

### 3. Exportar las hojas de anotación

Cada hoja trae `id`, la oración con sus vecinas como contexto y las columnas vacías que llenan los anotadores. Las estrellas, el restaurante, el estrato y lo que marca el regex quedan **solo** en `clave/muestreo.csv`, para no sesgar a quien anota.

In [6]:
def exportar(nombre, orac, idx, semilla):
    ruta = ANOTACION / nombre
    hoja = et.hoja_anotacion(orac, idx, semilla=semilla)
    if ruta.exists():
        actual = pd.read_csv(ruta, dtype=str, keep_default_na=False)
        assert set(actual.id) == set(hoja.id), f"{nombre}: el muestreo cambió respecto al archivo existente"
        print(f"{nombre:26} ya existe ({len(actual)} filas, {(actual.revisada.str.strip() != '').sum()} revisadas); no se toca")
    else:
        hoja.to_csv(ruta, index=False, encoding="utf-8")
        print(f"{nombre:26} creado ({len(hoja)} filas)")
    return hoja

h_mad = exportar("madrid_train.csv", madrid, clave_mad.idx, semilla=0)
h_bcn = exportar("bcn_gold.csv", bcn, clave_bcn.idx, semilla=0)
h_bcn2 = exportar("bcn_gold_anotador2.csv", bcn, sorted(doble), semilla=2)
h_extra = exportar("bcn_gold_extra.csv", bcn, clave_extra.idx, semilla=3)

madrid_train.csv           ya existe (300 filas, 300 revisadas); no se toca


bcn_gold.csv               ya existe (300 filas, 300 revisadas); no se toca


bcn_gold_anotador2.csv     ya existe (100 filas, 100 revisadas); no se toca


bcn_gold_extra.csv         ya existe (100 filas, 100 revisadas); no se toca


In [7]:
def clave(orac, c, archivo):
    sel = orac.loc[c.idx]
    out = pd.DataFrame({"id": (sel.review_id + "#" + sel.n.astype(str)).values, "archivo": archivo,
                        "grupo": c.grupo.values, "estrato": c.estrato.values, "N_estrato": c.N_estrato.values,
                        "peso": c.peso.values, "restaurante": sel.restaurante.values, "estrellas": sel.estrellas.values})
    reg = et.etiquetas_regex(sel.oracion).add_prefix("regex_")
    return pd.concat([out, reg.reset_index(drop=True)], axis=1)

muestreo = pd.concat([clave(madrid, clave_mad, "madrid_train.csv"),
                      clave(bcn, gold[gold.archivo == "bcn_gold.csv"], "bcn_gold.csv"),
                      clave(bcn, gold[gold.archivo == "bcn_gold_extra.csv"], "bcn_gold_extra.csv")],
                     ignore_index=True)
muestreo["doble"] = muestreo.id.isin(h_bcn2.id)
ruta_clave = CLAVE / "muestreo.csv"
if ruta_clave.exists():
    # Igual que con las hojas: la clave existente no se reescribe; se comprueba que el muestreo da lo mismo
    pd.testing.assert_frame_equal(pd.read_csv(ruta_clave), muestreo.reset_index(drop=True), check_dtype=False, check_exact=False)
    print("clave/muestreo.csv ya existe y coincide con el muestreo; no se toca")
else:
    muestreo.to_csv(ruta_clave, index=False)
    print("clave/muestreo.csv creado")
es_gold = muestreo.archivo.str.startswith("bcn_gold")
marca = muestreo[es_gold].filter(like="regex_").any(axis=1)
print(f"Oraciones del gold (base + extra) que el regex marca: {marca.sum()} de {es_gold.sum()} "
      "→ el resto forma el subconjunto sin léxico (paráfrasis)")
muestreo.groupby("archivo").size()

clave/muestreo.csv ya existe y coincide con el muestreo; no se toca
Oraciones del gold (base + extra) que el regex marca: 29 de 400 → el resto forma el subconjunto sin léxico (paráfrasis)


archivo
bcn_gold.csv          300
bcn_gold_extra.csv    100
madrid_train.csv      300
dtype: int64

In [8]:
h_bcn.head(5)

,id,anterior,oracion,siguiente,llegada,atencion,cocina_mesa,cobro,sabor,precio,local,dudosa,revisada,nota
0,review_220632428#2,Easy to book over the phone even if we did not speak Spanish.,Not cheap but you pay for location and ambience.,"Paella was tasty but certainly not ""out of the box"".",,,,,,,,,,
1,review_466143153#0,,"The food was too salty or too sweet (depends on the fish we ordered) and it was, unfortunately, extremely expensive.",The waiters could not speak English.,,,,,,,,,,
2,review_361848714#4,A really big big big Cockroach was under us coming from the other table which scarred us and the others.,"Unfortunately the system on this place is to pay before you eat your food otherwise , i think we shouldn't pay.",First and last time to be there.,,,,,,,,,,
3,review_665800340#1,"Really tasty food, and presentation is so pleasing to eyes!",You may queue over the weekend around lunch time - but it’s worth a visit.,,,,,,,,,,,
4,review_444106691#0,,"Went with my family for a mid day menu, and had several options to choose from.",They have a salad bar buffet option.,,,,,,,,,,


### 4. Al volver de la anotación: progreso y acuerdo

Estas celdas corren siempre. Mientras falten anotaciones, solo informan el avance. El κ de Cohen se calcula por etapa sobre las oraciones que los dos anotadores marcaron como `revisada`. La meta es κ ≥ 0.65. **El κ mide consistencia entre los dos anotadores, no validez.** Un κ bajo en una etapa significa que su definición en la guía es ambigua, no que alguien anotó mal.

In [9]:
for nombre in ["madrid_train.csv", "bcn_gold.csv", "bcn_gold_extra.csv", "bcn_gold_anotador2.csv"]:
    h = pd.read_csv(ANOTACION / nombre, dtype=str, keep_default_na=False)
    n_rev = (h.revisada.str.strip() != "").sum()
    print(f"{nombre:26} {n_rev:3d} / {len(h)} revisadas")

madrid_train.csv           300 / 300 revisadas
bcn_gold.csv               300 / 300 revisadas
bcn_gold_extra.csv         100 / 100 revisadas
bcn_gold_anotador2.csv     100 / 100 revisadas


In [10]:
a1 = et.leer_anotacion(ANOTACION / "bcn_gold.csv").set_index("id")
a2 = et.leer_anotacion(ANOTACION / "bcn_gold_anotador2.csv").set_index("id")
comunes = a1.index.intersection(a2.index)
if len(comunes) < 80:
    print(f"κ pendiente: hay {len(comunes)} oraciones revisadas por los dos anotadores (se necesitan ~100).")
else:
    filas = []
    for e in et.ETIQUETAS:
        x, y = a1.loc[comunes, e], a2.loc[comunes, e]
        filas.append({"etapa": e, "positivos A1": int(x.sum()), "positivos A2": int(y.sum()),
                      "acuerdo %": round((x == y).mean() * 100, 1),
                      "κ entre anotadores": round(cohen_kappa_score(x, y), 3) if (x.sum() + y.sum()) else np.nan})
    kappa = pd.DataFrame(filas).set_index("etapa")
    print(f"Oraciones comunes: {len(comunes)} | κ entre anotadores, medio en el ciclo: {kappa.loc[et.CICLO, 'κ entre anotadores'].mean():.3f} "
          f"(meta ≥ 0.65: consistencia, no validez)")
    display(kappa)
    desac = (a1.loc[comunes, et.ETIQUETAS] != a2.loc[comunes, et.ETIQUETAS]).any(axis=1)
    print(f"Oraciones con algún desacuerdo: {desac.sum()} (se reportan tal cual; el gold oficial es el del anotador 1 y ninguna etiqueta se edita)")
    display(a1.loc[comunes[desac], ["oracion"]].join(a1[et.ETIQUETAS].add_suffix("_A1")).join(a2[et.ETIQUETAS].add_suffix("_A2")))

Oraciones comunes: 100 | κ entre anotadores, medio en el ciclo: 0.949 (meta ≥ 0.65)


,positivos A1,positivos A2,acuerdo %,κ entre anotadores
etapa,,,,
llegada,0,0,100.0,NaN
atencion,10,10,100.0,1.000
cocina_mesa,7,8,99.0,0.928
cobro,6,7,99.0,0.918
sabor,16,18,96.0,0.858
precio,7,9,98.0,0.864
local,3,3,100.0,1.000


Oraciones con algún desacuerdo: 7 (se reportan tal cual; el gold oficial es el del anotador 1 y ninguna etiqueta se edita)


,oracion,llegada_A1,atencion_A1,cocina_mesa_A1,cobro_A1,sabor_A1,precio_A1,local_A1,llegada_A2,atencion_A2,cocina_mesa_A2,cobro_A2,sabor_A2,precio_A2,local_A2
id,,,,,,,,,,,,,,,
review_410345656#4,Not an award winning restaurant but a solid option if you need something in the area.,0,0,0,0,0,0,0,0,0,1,0,0,0,0
review_250397468#2,Sad really because the staff seemed nice and the food was OK - but not brilliant.,0,0,0,0,0,0,0,0,0,0,1,0,0,0
review_675944371#3,20 euros for mojito without alcohol ( water with mint and lime)?,0,0,0,0,0,1,0,0,0,0,0,1,1,0
review_271073434#6,We were obviously waiting for the second burger who took an additional 35 minutes to arrive and was terribly underco...,0,0,1,0,1,0,0,0,0,1,0,0,0,0
review_142525924#12,Setting isn't everything.,0,0,0,0,0,0,0,0,0,0,0,0,1,0
review_206885632#17,"The staff said no one else had a problem with the food, again this highlights their lack of customer service but als...",0,1,0,0,0,0,0,0,1,0,0,1,0,0
review_158715533#1,ur better off finding something local or even going to star bucks,0,0,0,0,0,0,0,0,0,0,0,1,1,0


### Cifras clave


In [11]:
gold_cl = muestreo[muestreo.archivo.str.startswith("bcn_gold")]
resultados.guardar("04", {
    "madrid_neg_oraciones": len(neg), "madrid_cobertura_oracion_pct": cubre * 100,
    "n_madrid_train": int((muestreo.archivo == "madrid_train.csv").sum()), "n_gold_base": int((muestreo.archivo == "bcn_gold.csv").sum()),
    "n_gold_extra": int((muestreo.archivo == "bcn_gold_extra.csv").sum()), "n_doble": int(muestreo.doble.sum()),
    "gold_regex_marca": int(gold_cl.filter(like="regex_").any(axis=1).sum()),
    "kappa": None if len(comunes) < 80 else kappa["κ entre anotadores"], "kappa_ciclo_medio": None if len(comunes) < 80 else kappa.loc[et.CICLO, "κ entre anotadores"].mean(),
})

hallazgos_04.json: 9 claves → gold_regex_marca, kappa, kappa_ciclo_medio, madrid_cobertura_oracion_pct, madrid_neg_oraciones, n_doble, n_gold_base, n_gold_extra, n_madrid_train


PosixPath('/Users/georgealbadr/Github/Natural_Language_Processing/resultados/hallazgos_04.json')

### Hallazgos

- **El regex del avance ve poco a nivel de oración:** marca solo el 10.4% de las 75,265 oraciones de reseñas 1-2★ de Madrid train. El resto son quejas parafraseadas, quejas generales u oraciones neutras; el gold dirá qué parte es cuál.
- **El gold de Barcelona es casi todo "sin léxico":** el regex marca 29 de las 400 oraciones (300 base + 100 extra). El resto es el subconjunto donde un modelo tiene que ganarle al regex.
- **Cobro y Llegada se sobre-muestrean sin usar el regex.** Con palabras de tema y la ampliación, 105 oraciones caen en "tema cobro" y 105 en "tema llegada", frente a ~13 y ~33 que daría un muestreo plano de 300. Los pesos `N/n` combinados corrigen eso en el 06.
- **Carga de anotación:** 800 oraciones en total (300 + 100 extra + 300 de Madrid + 100 dobles), repartidas entre tres anotadores independientes. La guía está en `data/anotacion/guia_anotacion.md`.
- **Pendiente:** κ entre anotadores (meta ≥ 0.65). Se calcula aquí mismo cuando vuelvan las hojas; los pasos 05 y 06 esperan esa anotación.

## 05 · Clasificador de fallas: etiquetas débiles y escalera de modelos

| | |
|---|---|
| **Qué entra** | oraciones de Madrid (00), léxico v2 (03), `data/anotacion/madrid_train.csv` (04) |
| **Qué sale** | modelos en `modelos/` (LogReg, MLP, RNN, LSTM y fastText v2) y la tabla de la escalera en dev |

Cada oración recibe 7 etiquetas binarias, una por etapa (las 4 del ciclo y 3 fuera de él), que valen 1 si reporta una falla de esa etapa. Todos los modelos se entrenan con lo mismo, se eligen en el **dev de Madrid** (restaurantes que no se usaron para entrenar) y se comparan en una sola tabla. El gold de Barcelona **no se toca aquí**: se usa una sola vez, en el 06.

**Estado:** final. Entrena con las etiquetas débiles y las 300 oraciones anotadas de madrid_train (peso ×5); la variante y la preferencia entre LSTM y LogReg las decide el dev de paráfrasis reales (sección 6), antes de mirar el gold.

In [1]:
import time

import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader
from foodie_advisor import MODELOS, FIGURAS, datos, lexico, etiquetas as et, modelos as m

sns.set_theme(style="whitegrid", font_scale=0.95)
plt.rcParams["font.family"] = "DejaVu Sans"
E, CICLO = et.ETIQUETAS, et.CICLO
np.random.seed(0); torch.manual_seed(42)
print("Dispositivo:", m.dispositivo())

Dispositivo: mps


### 1. Datos de entrenamiento

- **Etiquetas débiles (léxico v2 del 03):** una oración es positiva para una etapa si la marca el regex y el término no está negado. Los negativos son oraciones de reseñas de 5★ sin ninguna coincidencia. Hay hasta 6,000 positivos por etapa (las raras entran completas) y 30,000 negativos.
- **Enmascarado:** en la mitad de los positivos, el término que disparó el regex se reemplaza por `<mask>`. Así el modelo no puede limitarse a copiar el regex y tiene que aprender del contexto.
- **Anotadas:** las 300 oraciones de `madrid_train.csv` (1-2★, sin léxico) entran con peso ×5 cuando vuelven anotadas (por un anotador independiente; ver 04).

In [2]:
orac = datos.cargar_oraciones("Madrid", con_resena=True)
rest_tr, rest_dev = et.particion_restaurantes(orac.restaurante)
o_tr, o_dev = orac[orac.restaurante.isin(rest_tr)], orac[orac.restaurante.isin(rest_dev)]

debil = et.conjunto_debil(o_tr, etapas=lexico.ETAPAS_V2, guardia_negacion=True)
anot = et.anotadas("madrid_train.csv", peso=5.0)
train = pd.concat([debil, anot], ignore_index=True)

resumen = pd.DataFrame({"débiles": debil[E].sum(), "anotadas": anot[E].sum() if len(anot) else 0})
print(f"Train: {len(debil):,} débiles ({debil.enmascarada.sum():,} enmascaradas) + {len(anot)} anotadas | "
      f"{len(rest_tr)} restaurantes")
resumen.T

Train: 51,598 débiles (10,812 enmascaradas) + 300 anotadas | 1201 restaurantes


,llegada,atencion,cocina_mesa,cobro,sabor,precio,local
débiles,1099,5485,2905,702,4302,6091,2151
anotadas,6,71,19,10,52,16,6


In [3]:
n_lleg = int(anot["llegada"].sum()) if len(anot) else 0
LLEGADA_EN_MACRO = n_lleg >= 15
print(f"Positivos de Llegada en madrid_train anotado: {n_lleg} → "
      + ("entra al F1 macro del ciclo" if LLEGADA_EN_MACRO else
         "menos de 15: Llegada se reporta con su IC pero queda FUERA del F1 macro (regla fijada antes de evaluar)")
      + ("" if len(anot) else "  [anotación pendiente]"))

Positivos de Llegada en madrid_train anotado: 6 → menos de 15: Llegada se reporta con su IC pero queda FUERA del F1 macro (regla fijada antes de evaluar)


### 2. Dev de Madrid: completo y sin disparador

El dev sale de otros 213 restaurantes y se etiqueta con el mismo procedimiento, pero **sin** `<mask>`, porque en Barcelona no hay máscaras. Medimos dos versiones:

- **dev completo:** las oraciones tal cual. El regex v2 acierta aquí por construcción.
- **dev sin disparador:** a los positivos se les **borra** el término que los marcó; la celda de abajo imprime un ejemplo real del antes y el después. Es lo más parecido a una paráfrasis que se puede armar sin anotar: el regex ya no ve nada y el modelo tiene que reconocer la falla por el contexto.

Las dos son etiquetas débiles, así que sirven para **elegir** modelos, no para reportar su calidad. Eso lo hace el gold.

In [4]:
dev = et.conjunto_debil(o_dev, max_pos_por_etapa=1500, n_neg=8000, frac_mask=0.0, semilla=1,
                        etapas=lexico.ETAPAS_V2, guardia_negacion=True)
es_pos = dev[E].any(axis=1).values
dev_sin = dev.copy()
dev_sin.loc[es_pos, "texto"] = [et.quitar_disparador(x, lexico.ETAPAS_V2) for x in dev_sin.texto[es_pos]]
dev_sin = dev_sin[dev_sin.texto.str.split().str.len() >= 3]
print(f"dev completo: {len(dev):,} oraciones ({es_pos.sum():,} positivas) | dev sin disparador: {len(dev_sin):,}")
print("Ejemplo:\n  antes  :", dev.texto[es_pos].iloc[3], "\n  después:", dev_sin.texto[dev_sin[E].any(axis=1)].iloc[3])

NOMBRES_MACRO = CICLO if LLEGADA_EN_MACRO else [e for e in CICLO if e != "llegada"]
def f1_ciclo(Y, P):
    tabla, _, _ = m.metricas(Y, P, E)
    return tabla.loc[NOMBRES_MACRO, "F1"].mean()

def evaluar(nombre, P_dev, P_sin, **extra):
    return {"modelo": nombre, "F1 ciclo dev": f1_ciclo(dev[E].values, P_dev),
            "F1 ciclo sin disparador": f1_ciclo(dev_sin[E].values, P_sin),
            "F1 7 etapas dev": m.metricas(dev[E].values, P_dev, E)[2], **extra}
escalera = []

dev completo: 12,424 oraciones (4,424 positivas) | dev sin disparador: 12,321
Ejemplo:
  antes  : Slightly more expensive but an absolute must for anyone visiting Madrid. 
  después: slightly more but an absolute must for anyone visiting madrid.


### 3. fastText propio

Configuración base (skip-gram, d = 100, ventana 5, 10 épocas, semilla 42). Se entrena con **todas** las oraciones de los restaurantes de train y con las copias enmascaradas del v2, para que `<mask>` tenga vector propio. No entra nada del dev ni de Barcelona.

In [5]:
NOMBRE_FT = "fasttext_madrid_train_v2"
if (MODELOS / f"{NOMBRE_FT}.model").exists():
    ft = m.cargar_fasttext(NOMBRE_FT); print("fastText cargado de modelos/")
else:
    corpus = [m.tokens(x) for x in o_tr.oracion] + [m.tokens(x) for x in debil.texto[debil.enmascarada]]
    t0 = time.time(); ft = m.entrenar_fasttext(corpus, NOMBRE_FT)
    print(f"fastText: {len(corpus):,} oraciones, {time.time() - t0:.0f} s")
print(f"Vocabulario: {len(ft.wv):,}")
for w in ["rude", "<mask>", "bill", "waited"]:
    print(f"  {w:8} →", ", ".join(x for x, _ in ft.wv.most_similar(w, topn=8)))

fastText cargado de modelos/
Vocabulario: 32,965
  rude     → unfriendly, unpolite, impolite, arrogant, disrepectful, disrespectful, surly, dismissive
  <mask>   → bitchy, overprized, unfriendly, greaseless, overprice, rude, sloppy, lackluster
  bill     → tab, billed, gratuity, billled, calculate, billy, bills, paid
  waited   → waitng, awaited, waiting, wait, queued, waites, waived, elapsed


### 4. La escalera

#### 4.1 Mayoritaria y reglas

La clase mayoritaria (nunca hay falla) es el piso. Los dos regex son las reglas: el v1 es el del avance y el v2 el del 03.

In [6]:
cero = lambda d: np.zeros((len(d), len(E)))
escalera.append(evaluar("1. Clase mayoritaria", cero(dev), cero(dev_sin)))
for nombre, etapas, g in [("2a. Regex v1 (avance)", lexico.ETAPAS, False), ("2b. Regex v2 (03)", lexico.ETAPAS_V2, True)]:
    escalera.append(evaluar(nombre, et.etiquetas_regex(dev.texto, etapas, g).values,
                            et.etiquetas_regex(dev_sin.texto, etapas, g).values))
pd.DataFrame(escalera).set_index("modelo").round(3)

,F1 ciclo dev,F1 ciclo sin disparador,F1 7 etapas dev
modelo,,,
1. Clase mayoritaria,0.000,0.0,0.000
2a. Regex v1 (avance),0.914,0.0,0.928
2b. Regex v2 (03),1.000,0.0,1.000


#### 4.2 Naive Bayes y regresión logística sobre TF-IDF

Una por etapa (one-vs-rest), con unigramas y bigramas, sin quitar negadores. La C de la regresión logística se elige en dev. Probamos también la otra forma de esconder el disparador: en vez de `<mask>`, **borrarlo** del texto de entrenamiento. Con `<mask>`, el modelo puede aprender que "aquí había una palabra de queja", una pista que en el texto real no existe.

In [7]:
Y_tr, W_tr = train[E].values, train.peso.values
variantes_texto = {"<mask>": train.texto, "borrado": train.texto.str.replace(et.MASK, " ", regex=False)}
lineales = {}
for var, textos in variantes_texto.items():
    vec = m.tfidf().fit(textos)
    A, Bd, Bs = vec.transform(textos), vec.transform(dev.texto), vec.transform(dev_sin.texto)
    nb = m.nb_multietiqueta().fit(A, Y_tr)
    escalera.append(evaluar(f"3. Naive Bayes ({var})", nb.predict_proba(Bd), nb.predict_proba(Bs)))
    mejor = None
    for C in [0.3, 1, 3, 10]:
        lr = m.logreg_multietiqueta(C).fit(A, Y_tr)
        fila = evaluar(f"4. LogReg ({var})", lr.predict_proba(Bd), lr.predict_proba(Bs), C=C)
        sel = (fila["F1 ciclo dev"] + fila["F1 ciclo sin disparador"]) / 2
        if mejor is None or sel > mejor[0]:
            mejor = (sel, fila, lr)
    escalera.append(mejor[1]); lineales[var] = (vec, mejor[2], nb)
    print(f"{var}: C elegida = {mejor[1]['C']}")
pd.DataFrame(escalera).set_index("modelo").round(3)

<mask>: C elegida = 3


borrado: C elegida = 10


,F1 ciclo dev,F1 ciclo sin disparador,F1 7 etapas dev,C
modelo,,,,
1. Clase mayoritaria,0.000,0.000,0.000,NaN
2a. Regex v1 (avance),0.914,0.000,0.928,NaN
2b. Regex v2 (03),1.000,0.000,1.000,NaN
3. Naive Bayes (<mask>),0.582,0.092,0.624,NaN
4. LogReg (<mask>),0.869,0.425,0.884,3.0
3. Naive Bayes (borrado),0.591,0.112,0.635,NaN
4. LogReg (borrado),0.801,0.424,0.808,10.0


#### 4.3 Redes: MLP sobre el promedio de fastText, RNN y LSTM

Las tres usan la misma capa de embeddings, inicializada con el fastText de la sección 3 (`<pad>` = ceros, `<unk>` = vector aleatorio pequeño). La diferencia está en cómo resumen la oración:

- **MLP:** promedia los vectores de todas las palabras. El orden se pierde.
- **RNN / LSTM:** leen la oración palabra por palabra, de izquierda a derecha. El estado oculto después de la última palabra pasa a un `nn.Linear` con 7 salidas. Es un tagger secuencial, pero el clasificador se aplica una sola vez por oración y no una vez por palabra.

Para todas: `BCEWithLogitsLoss` con `pos_weight` = √(negativos / positivos) por etapa y Adam. Se guarda la época con mejor promedio del F1 del ciclo entre los dos dev.

In [8]:
def preparar(textos_train):
    vocab = m.vocabulario([m.tokens(x) for x in textos_train], min_frec=2)
    W = m.matriz_embeddings(vocab, ft)
    carga = lambda d, txt, sh: DataLoader(m.OracionesDataset(m.a_indices(txt, vocab), d[E].values, d.peso.values if "peso" in d else None),
                                          batch_size=64, shuffle=sh, collate_fn=m.collate)
    return vocab, W, carga

dev_eval = pd.concat([dev, dev_sin], ignore_index=True)
n_dev = len(dev)
sel = lambda P: (f1_ciclo(dev[E].values, P[:n_dev]) + f1_ciclo(dev_sin[E].values, P[n_dev:])) / 2
PW = np.sqrt(m.pos_weight_de(Y_tr))
print("pos_weight por etapa:", dict(zip(E, PW.round(1))))

redes, historiales = {}, {}
SEMILLAS = [42, 43, 44]
def correr(nombre, construir, var="<mask>", epocas=6):
    # Entrena con 3 semillas: la tabla reporta media ± desviación estándar (MPS no es determinista).
    # Se guarda el modelo de la semilla con mejor métrica de selección en dev.
    vocab, W, carga = preparar(variantes_texto[var])
    tr_l = carga(train.assign(texto=variantes_texto[var]), variantes_texto[var], True)
    dv_l = carga(dev_eval, dev_eval.texto, False)
    t0, filas, mejor = time.time(), [], None
    for s in SEMILLAS:
        torch.manual_seed(s)
        modelo = construir(W)
        h = m.entrenar(modelo, tr_l, dv_l, epocas=epocas, lr=0.002, pos_weight=PW, evaluar=sel, semilla=s, verbose=False)
        P = m.predecir_proba(modelo, dv_l)
        filas.append(evaluar(nombre, P[:n_dev], P[n_dev:]))
        if mejor is None or h["F1 macro dev"].max() > mejor[0]:
            mejor = (h["F1 macro dev"].max(), modelo, h)
    f = pd.DataFrame(filas)
    cols = ["F1 ciclo dev", "F1 ciclo sin disparador", "F1 7 etapas dev"]
    escalera.append({"modelo": nombre, **f[cols].mean().to_dict(), **{f"± {c}": f[c].std() for c in cols},
                     "épocas": int(mejor[2]["F1 macro dev"].idxmax() + 1), "segundos": round(time.time() - t0)})
    redes[nombre], historiales[nombre] = (mejor[1], vocab), mejor[2]
    print(f"{nombre}: {time.time() - t0:.0f} s | sin disparador {f['F1 ciclo sin disparador'].mean():.3f} ± {f['F1 ciclo sin disparador'].std():.3f}")

for var in ["<mask>", "borrado"]:
    correr(f"5. MLP promedio fastText ({var})", lambda W: m.MLPPromedio(W, len(E)), var=var)
    correr(f"6a. RNN ({var})", lambda W: m.Recurrente(W, len(E), celda="rnn"), var=var)
    correr(f"6b. LSTM ({var})", lambda W: m.Recurrente(W, len(E), celda="lstm"), var=var)

pos_weight por etapa: {'llegada': 6.8, 'atencion': 2.9, 'cocina_mesa': 4.1, 'cobro': 8.5, 'sabor': 3.3, 'precio': 2.7, 'local': 4.8}


5. MLP promedio fastText (<mask>): 41 s | sin disparador 0.423 ± 0.010


6a. RNN (<mask>): 232 s | sin disparador 0.180 ± 0.016


6b. LSTM (<mask>): 299 s | sin disparador 0.310 ± 0.005


5. MLP promedio fastText (borrado): 36 s | sin disparador 0.447 ± 0.005


6a. RNN (borrado): 233 s | sin disparador 0.322 ± 0.046


6b. LSTM (borrado): 256 s | sin disparador 0.487 ± 0.018


In [9]:
fig, ax = plt.subplots(figsize=(7, 3.4))
for nombre, h in historiales.items():
    ax.plot(h["época"], h["F1 macro dev"], marker="o", label=nombre)
ax.set(xlabel="Época", ylabel="F1 ciclo (promedio de los dos dev)", title="Curvas de validación")
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(FIGURAS / "05_curvas_dev.png", dpi=150); plt.show()

<Figure size 700x340 with 1 Axes>

### 5. La escalera en dev

**El dev completo mide acuerdo con el regex, no acierto real.** Sus etiquetas salen del regex v2, así que el v2 no se puede evaluar ahí (sería circular) y el resto de los modelos se mide por cuánto se parecen a él. El dev sin disparador es un proxy débil y solo sirve para seleccionar. Las redes se reportan como media ± desviación estándar de 3 semillas.

In [10]:
tabla = pd.DataFrame(escalera).set_index("modelo")
tabla = tabla.loc[sorted(tabla.index, key=lambda n: (n.split(". ")[0], "borrado" in n))]
# El v2 genera las etiquetas débiles: su "acierto" en dev completo es circular y no se evalúa
tabla.loc["2b. Regex v2 (03)", ["F1 ciclo dev", "F1 7 etapas dev"]] = np.nan
tabla["nota"] = ""
tabla.loc["2a. Regex v1 (avance)", "nota"] = "dev completo = acuerdo con el v2, no acierto real"
tabla.loc["2b. Regex v2 (03)", "nota"] = "fuente de las etiquetas: no evaluable en dev completo"
display(tabla.round(3))
comp = tabla[tabla.index.str.contains("<mask>|borrado")].copy()
comp["variante"] = comp.index.str.extract(r"\((.*)\)$", expand=False)
comp["modelo base"] = comp.index.str.replace(r" \(.*\)$", "", regex=True)
print("F1 ciclo en el dev SIN disparador, según cómo se escondió el término al entrenar:")
display(comp.pivot(index="modelo base", columns="variante", values="F1 ciclo sin disparador").round(3))

fig, ax = plt.subplots(figsize=(8, 4.2))
t_ = tabla[["F1 ciclo dev", "F1 ciclo sin disparador"]].iloc[::-1]
t_.plot.barh(ax=ax, color=["#9db7d5", "#b5483a"], width=0.8)
ax.set(xlabel=f"F1 macro del ciclo ({', '.join(NOMBRES_MACRO)})", ylabel="", title="Escalera de modelos en el dev de Madrid (etiquetas débiles)")
ax.legend(loc="lower right", fontsize=8); plt.tight_layout(); plt.savefig(FIGURAS / "05_escalera_dev.png", dpi=150); plt.show()

,F1 ciclo dev,F1 ciclo sin disparador,F1 7 etapas dev,C,± F1 ciclo dev,± F1 ciclo sin disparador,± F1 7 etapas dev,épocas,segundos,nota
modelo,,,,,,,,,,
1. Clase mayoritaria,0.000,0.000,0.000,NaN,NaN,NaN,NaN,NaN,NaN,
2a. Regex v1 (avance),0.914,0.000,0.928,NaN,NaN,NaN,NaN,NaN,NaN,"dev completo = acuerdo con el v2, no acierto real"
2b. Regex v2 (03),NaN,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fuente de las etiquetas: no evaluable en dev c...
3. Naive Bayes (<mask>),0.582,0.092,0.624,NaN,NaN,NaN,NaN,NaN,NaN,
3. Naive Bayes (borrado),0.591,0.112,0.635,NaN,NaN,NaN,NaN,NaN,NaN,
4. LogReg (<mask>),0.869,0.425,0.884,3.0,NaN,NaN,NaN,NaN,NaN,
4. LogReg (borrado),0.801,0.424,0.808,10.0,NaN,NaN,NaN,NaN,NaN,
5. MLP promedio fastText (<mask>),0.673,0.423,0.702,NaN,0.003,0.010,0.008,4.0,41.0,
5. MLP promedio fastText (borrado),0.656,0.447,0.684,NaN,0.013,0.005,0.005,5.0,36.0,


F1 ciclo en el dev SIN disparador, según cómo se escondió el término al entrenar:


variante,<mask>,borrado
modelo base,,
3. Naive Bayes,0.092,0.112
4. LogReg,0.425,0.424
5. MLP promedio fastText,0.423,0.447
6a. RNN,0.180,0.322
6b. LSTM,0.310,0.487


<Figure size 800x420 with 1 Axes>

**Cómo leer la tabla.** El dev completo mide **acuerdo con el regex, no acierto real**: el v1 llega a 0.914 y el v2 no se evalúa ahí, porque sus etiquetas son el propio v2. En el dev sin disparador, el regex saca 0 por construcción. Es un proxy débil, que solo sirve para seleccionar. Las redes se reportan como la media de 3 semillas (la desviación estándar va en las columnas ±). Todos los modelos ya incluyen las 300 oraciones anotadas de Madrid.

- **Naive Bayes** es el escalón más bajo (0.58-0.59 en dev, 0.09-0.11 sin disparador), igual que en SemEval (02).
- **La regresión logística** es la línea base fuerte: 0.869 en dev y 0.425 sin disparador con `<mask>`; 0.801 y 0.424 con el término borrado.
- **El MLP sobre el promedio de fastText** saca 0.447 sin disparador con el término borrado, gracias a `pos_weight`.
- **El RNN es la peor de las redes** (0.322 sin disparador con el término borrado). Aquí toda la oración tiene que llegar al último estado oculto, y la RNN simple pierde la información del principio.
- **El LSTM con `<mask>` sigue imitando al regex:** 0.906 en dev, cerca del regex v1 (0.914), pero solo 0.310 sin disparador.
- **El LSTM con el término borrado** es el mejor en el proxy: 0.487 sin disparador, 6 puntos sobre la regresión logística (0.424).

En el proxy, borrar le gana a `<mask>` en NB, MLP, RNN y LSTM, y empata en la regresión logística (0.424 contra 0.425). Como el proxy favorece al borrado por construcción, la decisión la toma el dev de paráfrasis reales (sección 6).

### 6. Dev de paráfrasis reales (decide la variante y LSTM contra LogReg)

El dev sin disparador es un proxy con un sesgo: se construye **borrando** el término, que es la misma transformación con la que se entrena la variante "borrado", así que esa variante juega con ventaja. Las 300 oraciones anotadas de `madrid_train` no tienen ese problema. Son paráfrasis naturales (reseñas de 1-2★ que el regex v1 no cubre), con etiquetas de un anotador independiente del regex.

Se hacen 5 particiones agrupadas por restaurante. En cada una se entrena con todo el conjunto débil más 4/5 de las anotadas (peso ×5) y se evalúa en el 1/5 restante. **Lo que decide es el promedio de las 5 corridas.** Si contradice al proxy, gana este dev.

In [11]:
from sklearn.model_selection import GroupKFold
clave = pd.read_csv("../data/anotacion/clave/muestreo.csv")
if len(anot) < 250:
    print(f"Pendiente: hay {len(anot)} oraciones de madrid_train revisadas (se necesitan ~300).")
    real = None
else:
    anot_r = anot.merge(clave[["id", "restaurante"]], on="id")
    epocas_lstm = {var: int(historiales[f"6b. LSTM ({var})"]["F1 macro dev"].idxmax() + 1) for var in variantes_texto}
    filas = []
    for k, (i_tr, i_te) in enumerate(GroupKFold(n_splits=5).split(anot_r, groups=anot_r.restaurante)):
        prueba = anot_r.iloc[i_te]
        for var in variantes_texto:
            txt_debil = debil.texto if var == "<mask>" else debil.texto.str.replace(et.MASK, " ", regex=False)
            tr_k = pd.concat([debil.assign(texto=txt_debil), anot_r.iloc[i_tr]], ignore_index=True)
            C = tabla.loc[f"4. LogReg ({var})", "C"]
            vec = m.tfidf().fit(tr_k.texto)
            lr = m.logreg_multietiqueta(C).fit(vec.transform(tr_k.texto), tr_k[E].values)
            P_lr = lr.predict_proba(vec.transform(prueba.texto))
            vocab, W, carga = preparar(tr_k.texto)
            lstm = m.Recurrente(W, len(E), celda="lstm")
            m.entrenar(lstm, carga(tr_k, tr_k.texto, True), epocas=epocas_lstm[var], pos_weight=PW, verbose=False)
            P_lstm = m.predecir_proba(lstm, carga(prueba, prueba.texto, False))
            for nombre, P in [("LogReg", P_lr), ("LSTM", P_lstm)]:
                tab, _, _ = m.metricas(prueba[E].values, P, E)
                filas.append({"partición": k + 1, "variante": var, "modelo": nombre,
                              "F1 ciclo": tab.loc[NOMBRES_MACRO, "F1"].mean(), "recall ciclo": tab.loc[NOMBRES_MACRO, "recall"].mean()})
    real = pd.DataFrame(filas)
    display(real.groupby(["variante", "modelo"])[["F1 ciclo", "recall ciclo"]].agg(["mean", "std"]).round(3))

# Decisión (antes de mirar el gold): la variante con mejor F1 medio de LSTM y LogReg en el dev real,
# y dentro de ella, LSTM si supera a LogReg en F1 del ciclo; si no, LogReg.
if real is None:
    VARIANTE, MODELO_DEV = "borrado", "LSTM"           # provisional, por el proxy
else:
    medias = real.groupby(["variante", "modelo"])["F1 ciclo"].mean()
    VARIANTE = medias.groupby("variante").mean().idxmax()
    MODELO_DEV = "LSTM" if medias[(VARIANTE, "LSTM")] > medias[(VARIANTE, "LogReg")] else "LogReg"
print(f"Decisión con el dev {'real' if real is not None else 'proxy (provisional)'}: variante = {VARIANTE}, modelo preferido = {MODELO_DEV}")

F1 ciclo        recall ciclo       
                    mean    std         mean    std
variante modelo                                    
<mask>   LSTM      0.295  0.231        0.289  0.299
         LogReg    0.295  0.184        0.281  0.225
borrado  LSTM      0.422  0.206        0.521  0.315
         LogReg    0.283  0.115        0.374  0.212

Decisión con el dev real: variante = borrado, modelo preferido = LSTM


**El dev de paráfrasis reales confirma al proxy: término borrado y LSTM.** En el promedio de las 5 particiones, el LSTM con el término borrado saca 0.422 ± 0.206 de F1 del ciclo y 0.521 de recall, contra 0.283 ± 0.115 y 0.374 de la regresión logística. Con `<mask>`, los dos quedan en 0.295. La desviación es grande porque cada partición tiene unas 60 oraciones y pocas son positivas; la diferencia entre el LSTM y la regresión logística es menor que una desviación estándar. La decisión (variante = borrado, preferido = LSTM) queda escrita en `hallazgos_05.json` antes de abrir el gold.

### 7. Guardar los modelos para el 06

Se guardan todos los escalones, porque la tabla del 06 compara la escalera completa contra el gold, y además el LSTM y la regresión logística elegidos. Ninguno vio Barcelona.

In [12]:
for var, (vec, lr, nb) in lineales.items():
    joblib.dump({"vec": vec, "logreg": lr, "nb": nb}, MODELOS / f"lineales_{'mask' if var == '<mask>' else 'borrado'}.joblib")
for nombre, (modelo, vocab) in redes.items():
    clave = nombre.split(". ")[1].replace(" ", "_").replace("<", "").replace(">", "").replace("(", "").replace(")", "")
    m.guardar_red(modelo.cpu(), vocab, clave, modelo_escalera=nombre, etiquetas=E)
tabla.to_csv(MODELOS / "escalera_dev.csv")
print(sorted(p.name for p in MODELOS.iterdir()))

['LSTM_borrado.pt', 'LSTM_mask.pt', 'MLP_promedio_fastText_borrado.pt', 'MLP_promedio_fastText_mask.pt', 'RNN_borrado.pt', 'RNN_mask.pt', 'escalera_dev.csv', 'fasttext_madrid_train_v2.model', 'fasttext_madrid_train_v2.model.wv.vectors_ngrams.npy', 'lineales_borrado.joblib', 'lineales_mask.joblib']


### Cifras clave

In [13]:
resultados.guardar("05", {
    "tabla_dev": tabla.drop(columns=["nota"]), "pivot_sin_disparador": comp.pivot(index="modelo base", columns="variante", values="F1 ciclo sin disparador"),
    "llegada_positivos_anotados": n_lleg, "llegada_en_macro": LLEGADA_EN_MACRO, "anotadas_n": len(anot),
    "debiles_n": len(debil), "enmascaradas_n": int(debil.enmascarada.sum()), "positivos_debiles": debil[E].sum(),
    "dev_n": len(dev), "dev_sin_n": len(dev_sin), "semillas": SEMILLAS,
    "dev_real": None if real is None else real.groupby(["variante", "modelo"])[["F1 ciclo", "recall ciclo"]].mean(),
    "variante_elegida": VARIANTE, "modelo_preferido_dev": MODELO_DEV, "decidido_por_dev_real": real is not None,
})

hallazgos_05.json: 15 claves → anotadas_n, debiles_n, decidido_por_dev_real, dev_n, dev_real, dev_sin_n, enmascaradas_n, llegada_en_macro, llegada_positivos_anotados, modelo_preferido_dev, pivot_sin_disparador, positivos_debiles, semillas, tabla_dev, variante_elegida


PosixPath('/Users/georgealbadr/Github/Natural_Language_Processing/resultados/hallazgos_05.json')

### Hallazgos

- **Dev de paráfrasis reales** (300 oraciones anotadas de Madrid, 5 particiones por restaurante): el LSTM con el término borrado saca 0.422 ± 0.206 de F1 del ciclo, contra 0.283 ± 0.115 de la regresión logística. Decide: término borrado y LSTM preferido, antes del gold. La varianza es alta.
- **En el proxy** (dev sin disparador, débil), con el término borrado: NB 0.112, LogReg 0.424, MLP 0.447, RNN 0.322 y LSTM 0.487.
- **El LSTM con `<mask>` imita al regex:** 0.906 en dev completo, pero 0.310 sin disparador.
- **El RNN queda muy por debajo del LSTM** (0.322 contra 0.487): clasificar una oración entera exige memoria larga.
- **Llegada tiene 6 positivos en `madrid_train`:** por la regla de menos de 15, queda fuera del F1 macro.

### Verificación de citas

Toda oración citada entre comillas en este notebook tiene que existir literalmente en el corpus.

In [14]:
citas = []
verificadas = datos.verificar_citas(citas, ('Madrid',))
print(f"{len(verificadas)} citas verificadas:", verificadas)

0 citas verificadas: {}


## 05b · Calibración del umbral de LogReg (post-hoc, preparado sin tocar el gold)

| | |
|---|---|
| **Qué entra** | las 300 oraciones anotadas de `madrid_train.csv`, el conjunto débil de Madrid (05) y las predicciones de LogReg en Barcelona (07) |
| **Qué sale** | `resultados/hallazgos_05b.json`: un umbral por etapa (máximo F1 y precisión ≥ 0.6) y la tasa base de reseñas marcadas en 5★ |

**Contexto.** En el gold (06), la regresión logística con umbral 0.5 tiene una precisión del ciclo de ~0.42: marca de más. El umbral se fijó antes del gold y **no se cambia sobre el gold**. Aquí se calcula un umbral alternativo usando **solo** el dev de paráfrasis reales de Madrid (predicciones fuera de partición, con las mismas 5 particiones por restaurante del 05). Si se aprueba, entra al 06 como una fila aparte rotulada **post-hoc**.

**Ojo con la transferencia:** `madrid_train` son oraciones de reseñas 1-2★ que el regex no cubre, así que sus fallas son más frecuentes que en el gold de Barcelona (que también tiene 4-5★). Un umbral calibrado aquí puede quedar corto en precisión allá.

In [1]:
from sklearn.metrics import precision_recall_fscore_support
from foodie_advisor import ANOTACION, datos, lexico, etiquetas as et, modelos as m, resultados, scorecard as sc

E, CICLO = et.ETIQUETAS, et.CICLO
h05 = resultados.cargar("05")
C = h05["tabla_dev"]["4. LogReg (borrado)"]["C"]
orac = datos.cargar_oraciones("Madrid", con_resena=True)
rest_tr, _ = et.particion_restaurantes(orac.restaurante)
debil = et.conjunto_debil(orac[orac.restaurante.isin(rest_tr)], etapas=lexico.ETAPAS_V2, guardia_negacion=True)
debil = debil.assign(texto=debil.texto.str.replace(et.MASK, " ", regex=False))       # variante borrado
clave = pd.read_csv(ANOTACION / "clave" / "muestreo.csv")
anot = et.anotadas("madrid_train.csv", peso=5.0).merge(clave[["id", "restaurante"]], on="id")
print(f"LogReg C = {C} | débiles {len(debil):,} | anotadas {len(anot)} | positivos anotados:", anot[E].sum().to_dict())

LogReg C = 10.0 | débiles 51,598 | anotadas 300 | positivos anotados: {'llegada': 6, 'atencion': 71, 'cocina_mesa': 19, 'cobro': 10, 'sabor': 52, 'precio': 16, 'local': 6}


### 1. Probabilidades fuera de partición en las 300 anotadas

In [2]:
P_oof = np.zeros((len(anot), len(E)))
for i_tr, i_te in GroupKFold(n_splits=5).split(anot, groups=anot.restaurante):
    tr = pd.concat([debil, anot.iloc[i_tr]], ignore_index=True)
    vec = m.tfidf().fit(tr.texto)
    lr = m.logreg_multietiqueta(C).fit(vec.transform(tr.texto), tr[E].values, )
    P_oof[i_te] = lr.predict_proba(vec.transform(anot.texto.iloc[i_te]))
Y = anot[E].values
print("Listo: una probabilidad por oración anotada y etapa, siempre de un modelo que no la vio.")

Listo: una probabilidad por oración anotada y etapa, siempre de un modelo que no la vio.


### 2. Umbral por etapa

Para cada etapa se recorren los umbrales de 0.05 a 0.95.

**Criterio de la fila post-hoc del 06 (escrito antes de evaluarla):** umbral por etapa = el menor que da precisión ≥ 0.80 en este dev de paráfrasis reales de Madrid; si ninguno la alcanza, se usa el que maximiza F1. Se evalúa una sola vez en el gold y no cambia el scorecard.

Como referencia también se muestran el umbral de máximo F1 y el de precisión ≥ 0.6. Con pocos positivos (Llegada tiene 6), cualquier umbral es poco confiable.

In [3]:
grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
filas = []
for j, e in enumerate(E):
    pr = [precision_recall_fscore_support(Y[:, j], P_oof[:, j] >= u, average="binary", zero_division=0)[:3] for u in grid]
    pr = np.array(pr)
    u_f1 = grid[pr[:, 2].argmax()]
    ok = np.flatnonzero(pr[:, 0] >= 0.6)
    u_p60 = grid[ok[0]] if len(ok) else np.nan
    ok80 = np.flatnonzero(pr[:, 0] >= 0.8)
    u_post = grid[ok80[0]] if len(ok80) else u_f1
    p05 = precision_recall_fscore_support(Y[:, j], P_oof[:, j] >= 0.5, average="binary", zero_division=0)[:3]
    filas.append({"etapa": e, "positivos": int(Y[:, j].sum()), "P@0.5": p05[0], "R@0.5": p05[1], "F1@0.5": p05[2],
                  "umbral F1 máx": u_f1, "F1 en ese umbral": pr[:, 2].max(),
                  "umbral precisión ≥ 0.6": u_p60, "recall en ese umbral": pr[ok[0], 1] if len(ok) else np.nan,
                  "umbral post-hoc": u_post, "regla post-hoc": "precisión ≥ 0.80" if len(ok80) else "máximo F1 (ninguno llega a 0.80)"})
cal = pd.DataFrame(filas).set_index("etapa")
cal.round(3)

,positivos,P@0.5,R@0.5,F1@0.5,umbral F1 máx,F1 en ese umbral,umbral precisión ≥ 0.6,recall en ese umbral,umbral post-hoc,regla post-hoc
etapa,,,,,,,,,,
llegada,6,0.167,0.167,0.167,0.30,0.235,NaN,NaN,0.30,máximo F1 (ninguno llega a 0.80)
atencion,71,0.443,0.606,0.512,0.65,0.534,0.80,0.437,0.95,precisión ≥ 0.80
cocina_mesa,19,0.222,0.316,0.261,0.70,0.333,0.90,0.158,0.70,máximo F1 (ninguno llega a 0.80)
cobro,10,0.222,0.200,0.211,0.15,0.318,NaN,NaN,0.15,máximo F1 (ninguno llega a 0.80)
sabor,52,0.532,0.481,0.505,0.50,0.505,0.70,0.385,0.95,precisión ≥ 0.80
precio,16,0.200,0.438,0.275,0.90,0.522,0.85,0.375,0.90,precisión ≥ 0.80
local,6,0.111,0.167,0.133,0.80,0.286,0.80,0.167,0.80,precisión ≥ 0.80


### 3. Tasa base de reseñas marcadas en Barcelona (fuera del gold)

¿Qué parte de las reseñas de **5★** marca la regresión logística con alguna falla del ciclo? Es el piso de falsos positivos que ve el gerente. Se compara con 1-2★, con el regex y con los umbrales calibrados.

In [4]:
res = datos.cargar_resenas("Barcelona", ["review_id", "estrellas"])
grupo = pd.cut(res.estrellas, [0, 2, 3, 4, 5], labels=["1-2★", "3★", "4★", "5★"]).astype(str).values
filas = {}
for fuente in ["regex_v1", "LogReg_borrado"]:
    pr = pd.read_parquet(sc.ruta_pred(fuente, "Barcelona")).groupby("review_id")[CICLO].max().any(axis=1)
    filas[f"{fuente} (umbral 0.5)" if fuente != "regex_v1" else "regex v1"] = pr.reindex(res.review_id).fillna(False).astype(bool).groupby(grupo).mean() * 100

# Umbrales calibrados: hay que recalcular probabilidades (solo oraciones de Barcelona, nunca el gold)
lin = m.cargar_lineales("borrado")
o_b = datos.cargar_oraciones("Barcelona")
Pb = np.vstack([lin["logreg"].predict_proba(lin["vec"].transform(o_b.oracion.iloc[i:i + 400_000])) for i in range(0, len(o_b), 400_000)])
ci = [E.index(e) for e in CICLO]
for nombre, col in [("LogReg (umbral F1 máx, Madrid)", "umbral F1 máx"), ("LogReg (umbral precisión ≥ 0.6, Madrid)", "umbral precisión ≥ 0.6"),
                    ("LogReg (umbral post-hoc, Madrid)", "umbral post-hoc")]:
    u = cal.loc[CICLO, col].fillna(1.01).values
    marca = pd.Series((Pb[:, ci] >= u).any(axis=1)).groupby(o_b.review_id.values).max()
    filas[nombre] = marca.reindex(res.review_id).fillna(False).astype(bool).groupby(grupo).mean() * 100
base_fp = pd.DataFrame(filas).T
print("% de reseñas de Barcelona con alguna falla del ciclo marcada, por estrellas:")
base_fp.round(1)

/var/folders/p6/_x7dxjn130z553sf_dpn33wc0000gn/T/ipykernel_25537/1420434236.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  filas[f"{fuente} (umbral 0.5)" if fuente != "regex_v1" else "regex v1"] = pr.reindex(res.review_id).fillna(False).astype(bool).groupby(grupo).mean() * 100


/var/folders/p6/_x7dxjn130z553sf_dpn33wc0000gn/T/ipykernel_25537/1420434236.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  filas[f"{fuente} (umbral 0.5)" if fuente != "regex_v1" else "regex v1"] = pr.reindex(res.review_id).fillna(False).astype(bool).groupby(grupo).mean() * 100


/var/folders/p6/_x7dxjn130z553sf_dpn33wc0000gn/T/ipykernel_25537/1420434236.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  filas[nombre] = marca.reindex(res.review_id).fillna(False).astype(bool).groupby(grupo).mean() * 100


/var/folders/p6/_x7dxjn130z553sf_dpn33wc0000gn/T/ipykernel_25537/1420434236.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  filas[nombre] = marca.reindex(res.review_id).fillna(False).astype(bool).groupby(grupo).mean() * 100


% de reseñas de Barcelona con alguna falla del ciclo marcada, por estrellas:


/var/folders/p6/_x7dxjn130z553sf_dpn33wc0000gn/T/ipykernel_25537/1420434236.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  filas[nombre] = marca.reindex(res.review_id).fillna(False).astype(bool).groupby(grupo).mean() * 100


,1-2★,3★,4★,5★
regex v1,27.4,14.6,6.3,3.1
LogReg_borrado (umbral 0.5),84.3,64.3,37.4,27.5
"LogReg (umbral F1 máx, Madrid)",81.9,61.6,38.4,29.2
"LogReg (umbral precisión ≥ 0.6, Madrid)",59.7,33.2,11.5,5.6
"LogReg (umbral post-hoc, Madrid)",72.6,53.2,34.4,25.5


In [5]:
resultados.guardar("05b", {"C": C, "calibracion": cal, "umbrales_f1": cal["umbral F1 máx"], "umbrales_p60": cal["umbral precisión ≥ 0.6"], "umbrales_posthoc": cal["umbral post-hoc"], "regla_posthoc": cal["regla post-hoc"],
                           "flagging_barcelona": base_fp, "aviso": "post-hoc; calibrado solo en madrid_train (1-2★ sin léxico); no evaluado en el gold"})

hallazgos_05b.json: 8 claves → C, aviso, calibracion, flagging_barcelona, regla_posthoc, umbrales_f1, umbrales_p60, umbrales_posthoc


PosixPath('/Users/georgealbadr/Github/Natural_Language_Processing/resultados/hallazgos_05b.json')

## 06 · Evaluación final en el gold de Barcelona (una sola vez)

| | |
|---|---|
| **Qué entra** | `bcn_gold.csv` + `bcn_gold_extra.csv` (anotador 1), `clave/muestreo.csv` (pesos), los modelos del 05 y su decisión en dev (`hallazgos_05.json`) |
| **Qué sale** | P/R/F1 por etapa, el subconjunto de paráfrasis, umbrales, modelo elegido, sensibilidad y especificidad para el scorecard (`hallazgos_06.json`) |

**Declaración:** el gold fue anotado siguiendo rigurosamente la guía por anotadores independientes: se anotaron las 400 oraciones de prueba de Barcelona y se repitieron 100 de ellas para evaluar el acuerdo inter-anotador (κ = 0.949 en el ciclo de servicio). Adicionalmente se anotaron 300 oraciones de entrenamiento de Madrid.

**El subconjunto de paráfrasis lo define el regex v1** (oraciones donde no marca nada). El entrenamiento usa etiquetas del v2, que en ese subconjunto recupera muy poco; así que el subconjunto mide lo que queda fuera del léxico del avance, no de todo léxico.

**Reglas fijadas antes de mirar el gold:**
- La variante y la preferencia entre LSTM y LogReg vienen del dev de paráfrasis reales del 05.
- El umbral de decisión es 0.5 para todos los modelos.
- Ningún modelo se reentrena ni se ajusta después de esta evaluación.
- Las métricas principales usan los pesos del muestreo (N/n de cada estrato), así que valen para Barcelona entera; las métricas sin pesos van al lado.
- Llegada queda fuera del F1 macro si `madrid_train` tiene menos de 15 positivos de Llegada, y se reporta con su IC.

In [1]:
from foodie_advisor import ANOTACION, FIGURAS, datos, lexico, etiquetas as et, modelos as m, resultados

E, CICLO = et.ETIQUETAS, et.CICLO
h05, h04 = resultados.cargar("05"), resultados.cargar("04")
VARIANTE, MODELO_DEV = h05["variante_elegida"], h05["modelo_preferido_dev"]
MACRO = CICLO if h05["llegada_en_macro"] else [e for e in CICLO if e != "llegada"]
print(f"Decisión del 05 (dev {'real' if h05['decidido_por_dev_real'] else 'proxy'}): variante = {VARIANTE}, preferido = {MODELO_DEV}")
print(f"F1 macro sobre: {MACRO}  (Llegada en madrid_train: {h05['llegada_positivos_anotados']} positivos)")

clave = pd.read_csv(ANOTACION / "clave" / "muestreo.csv")
gold = pd.concat([et.anotadas("bcn_gold.csv"), et.anotadas("bcn_gold_extra.csv")], ignore_index=True)
gold = gold.drop(columns=["peso", "origen", "enmascarada"])   # el peso que importa aquí es el de muestreo (clave)
gold = gold.merge(clave.drop(columns=["archivo"]), on="id", validate="1:1")
dudosa = pd.concat([et.leer_anotacion(ANOTACION / f) for f in ["bcn_gold.csv", "bcn_gold_extra.csv"]]).set_index("id").dudosa
gold["dudosa"] = gold.id.map(dudosa)
gold["parafrasis"] = ~gold.filter(like="regex_").any(axis=1)
W, Y = gold.peso.values, gold[E].values
print(f"Gold: {len(gold)} oraciones | paráfrasis (el regex v1 no marca nada): {gold.parafrasis.sum()} | dudosas: {gold.dudosa.sum()}")
pd.DataFrame({"positivos": gold[E].sum(), "en paráfrasis": gold.loc[gold.parafrasis, E].sum(),
              "positivos ponderados (% de Barcelona)": [round(np.average(gold[e], weights=W) * 100, 2) for e in E]})

Decisión del 05 (dev real): variante = borrado, preferido = LSTM
F1 macro sobre: ['atencion', 'cocina_mesa', 'cobro']  (Llegada en madrid_train: 6 positivos)
Gold: 400 oraciones | paráfrasis (el regex v1 no marca nada): 371 | dudosas: 34


,positivos,en paráfrasis,positivos ponderados (% de Barcelona)
llegada,6,5,0.25
atencion,69,53,5.49
cocina_mesa,25,22,1.84
cobro,32,31,1.00
sabor,49,41,4.63
precio,26,21,1.13
local,11,9,2.91


### 1. Las reglas, escritas antes de evaluar

Hay dos reglas previas, y se aplican las dos.

**Selección** (qué modelo aprendido es el mejor):

1. El LSTM es el modelo final si cumple **todos** los umbrales: F1 macro del ciclo ≥ 0.70, precisión de Atención ≥ 0.80, recall en paráfrasis al menos 10 puntos sobre el regex v1, y le gana a LogReg en el subconjunto de paráfrasis.
2. Si no, el scorecard usa **el modelo más simple que cumpla** los tres primeros umbrales, en este orden: regex v1, regex v2, NB, LogReg, MLP, RNN, LSTM.
3. Si ninguno los cumple, se usa LogReg y se reporta tal cual.

**Aptitud** (si un modelo sirve para un gerente): F1 macro del ciclo ≥ 0.70 y precisión de Atención ≥ 0.80. Si ningún modelo aprendido es apto, el producto usa el baseline que sí cumple la precisión.

El κ no es un umbral que "se cumple": se reporta como consistencia entre anotadores.

In [2]:
ORDEN_SIMPLE = ["Regex v1", "Regex v2", "Naive Bayes", "LogReg", "MLP", "RNN", "LSTM"]

def cumple(f):
    return (f["F1 ciclo"] >= 0.70) and (f["precisión atención"] >= 0.80) and (f["recall paráfrasis"] >= f_regex_par + 0.10)

def elegir(tabla):
    lstm, lr = tabla.loc["LSTM"], tabla.loc["LogReg"]
    if cumple(lstm) and lstm["F1 ciclo paráfrasis"] > lr["F1 ciclo paráfrasis"]:
        return "LSTM", "cumple todos los umbrales y le gana a LogReg en paráfrasis"
    for nombre in ORDEN_SIMPLE:
        if cumple(tabla.loc[nombre]):
            return nombre, "el más simple que cumple los umbrales"
    return "LogReg", "ningún modelo cumple todos los umbrales; se usa LogReg y se reporta"
print("Regla fijada.")

Regla fijada.


### 2. Predicciones de toda la escalera sobre el gold

In [3]:
txt = gold.oracion
lin = m.cargar_lineales(VARIANTE)
clave_red = lambda base: f"{base}_{'mask' if VARIANTE == '<mask>' else VARIANTE}"
P = {"Mayoritaria": np.zeros_like(Y, dtype=float),
     "Regex v1": et.etiquetas_regex(txt, lexico.ETAPAS).values.astype(float),
     "Regex v2": et.etiquetas_regex(txt, lexico.ETAPAS_V2, guardia_negacion=True).values.astype(float),
     "Naive Bayes": lin["nb"].predict_proba(lin["vec"].transform(txt)),
     "LogReg": lin["logreg"].predict_proba(lin["vec"].transform(txt))}
for nombre, base in [("MLP", "MLP_promedio_fastText"), ("RNN", "RNN"), ("LSTM", "LSTM")]:
    red, vocab = m.cargar_red(clave_red(base))
    P[nombre] = m.proba_red(red, vocab, txt)
print({k: v.shape for k, v in P.items()})

{'Mayoritaria': (400, 7), 'Regex v1': (400, 7), 'Regex v2': (400, 7), 'Naive Bayes': (400, 7), 'LogReg': (400, 7), 'MLP': (400, 7), 'RNN': (400, 7), 'LSTM': (400, 7)}


### 3. La escalera en el gold

In [4]:
par = gold.parafrasis.values

def fila(nombre, Pm):
    tab, _, _ = m.metricas(Y, Pm, E, pesos=W)
    tab_p, _, _ = m.metricas(Y[par], Pm[par], E, pesos=W[par])
    tab_np, _, _ = m.metricas(Y, Pm, E)
    Yh = (Pm >= 0.5)
    ciclo_idx = [E.index(e) for e in MACRO]
    pos_par = Y[par][:, ciclo_idx] == 1
    rec_par = np.average(Yh[par][:, ciclo_idx][pos_par], weights=np.repeat(W[par][:, None], len(ciclo_idx), 1)[pos_par])
    return {"modelo": nombre, "F1 ciclo": tab.loc[MACRO, "F1"].mean(), "F1 ciclo sin pesos": tab_np.loc[MACRO, "F1"].mean(),
            "precisión atención": tab.loc["atencion", "precisión"], "recall atención": tab.loc["atencion", "recall"],
            "recall paráfrasis": rec_par, "F1 ciclo paráfrasis": tab_p.loc[MACRO, "F1"].mean(),
            "F1 7 etapas": tab.F1.mean(), **{f"F1 {e}": tab.loc[e, "F1"] for e in E}}

escalera = pd.DataFrame([fila(k, v) for k, v in P.items()]).set_index("modelo")
f_regex_par = escalera.loc["Regex v1", "recall paráfrasis"]
display(escalera[["F1 ciclo", "F1 ciclo sin pesos", "precisión atención", "recall paráfrasis", "F1 ciclo paráfrasis", "F1 7 etapas"]].round(3))
display(escalera[[f"F1 {e}" for e in E]].round(3))

,F1 ciclo,F1 ciclo sin pesos,precisión atención,recall paráfrasis,F1 ciclo paráfrasis,F1 7 etapas
modelo,,,,,,
Mayoritaria,0.000,0.000,0.000,0.000,0.000,0.000
Regex v1,0.114,0.113,1.000,0.000,0.000,0.171
Regex v2,0.133,0.129,1.000,0.027,0.027,0.192
Naive Bayes,0.131,0.174,0.702,0.125,0.100,0.157
LogReg,0.417,0.465,0.422,0.518,0.397,0.322
MLP,0.403,0.514,0.538,0.489,0.373,0.329
RNN,0.340,0.374,0.409,0.416,0.327,0.271
LSTM,0.390,0.464,0.491,0.462,0.371,0.335


,F1 llegada,F1 atencion,F1 cocina_mesa,F1 cobro,F1 sabor,F1 precio,F1 local
modelo,,,,,,,
Mayoritaria,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Regex v1,0.000,0.149,0.146,0.047,0.317,0.419,0.121
Regex v2,0.000,0.205,0.146,0.047,0.415,0.409,0.121
Naive Bayes,0.000,0.356,0.038,0.000,0.430,0.275,0.000
LogReg,0.164,0.494,0.215,0.542,0.301,0.187,0.353
MLP,0.034,0.577,0.138,0.492,0.499,0.137,0.424
RNN,0.071,0.448,0.209,0.363,0.376,0.301,0.129
LSTM,0.207,0.521,0.215,0.435,0.402,0.172,0.392


In [5]:
# IC 95%: bootstrap estratificado (se remuestrea dentro de cada estrato, conservando su tamaño)
rng = np.random.default_rng(0)
estratos = gold.groupby(["grupo", "estrato"]).indices
def bootstrap(Pm, reps=1000):
    out = []
    for _ in range(reps):
        idx = np.concatenate([rng.choice(ix, len(ix), replace=True) for ix in estratos.values()])
        tab, _, _ = m.metricas(Y[idx], Pm[idx], E, pesos=W[idx])
        out.append([tab.loc[MACRO, "F1"].mean()] + [tab.loc[e, "F1"] for e in E])
    return pd.DataFrame(out, columns=["F1 ciclo"] + E).quantile([0.025, 0.975]).T

ic = {k: bootstrap(P[k]) for k in ["Regex v1", "LogReg", "LSTM"]}
pd.concat(ic, axis=1).round(3)

Regex v1        LogReg          LSTM       
               0.025  0.975  0.025  0.975  0.025  0.975
F1 ciclo       0.037  0.202  0.314  0.523  0.299  0.478
llegada        0.000  0.000  0.036  0.377  0.000  0.546
atencion       0.026  0.295  0.313  0.657  0.343  0.678
cocina_mesa    0.000  0.351  0.069  0.403  0.062  0.434
cobro          0.000  0.156  0.364  0.715  0.273  0.620
sabor          0.118  0.484  0.187  0.442  0.238  0.588
precio         0.039  0.668  0.076  0.332  0.056  0.356
local          0.000  0.468  0.014  0.739  0.087  0.731

#### LogReg contra LSTM, pareado

La misma remuestra para los dos modelos. Si el IC de la diferencia incluye 0, los dos **no son distinguibles** en este gold.

In [6]:
def f1_macro(Pm, idx, sub=None):
    ix = idx if sub is None else idx[sub[idx]]
    tab, _, _ = m.metricas(Y[ix], Pm[ix], E, pesos=W[ix])
    return tab.loc[MACRO, "F1"].mean()

def pareado(fun, reps=1000):
    r2 = np.random.default_rng(1); d = []
    for _ in range(reps):
        idx = np.concatenate([r2.choice(ix, len(ix), replace=True) for ix in estratos.values()])
        d.append(fun(P["LogReg"], idx) - fun(P["LSTM"], idx))
    lo, hi = np.percentile(d, [2.5, 97.5])
    return {"diferencia (LogReg − LSTM)": np.mean(d), "IC inf": lo, "IC sup": hi, "distinguibles": not (lo <= 0 <= hi)}

dif = pd.DataFrame({"F1 ciclo": pareado(f1_macro),
                    "F1 ciclo paráfrasis": pareado(lambda Pm, idx: f1_macro(Pm, idx, sub=par))}).T
dif.round(3)

,diferencia (LogReg − LSTM),IC inf,IC sup,distinguibles
F1 ciclo,0.028323,-0.046476,0.098009,False
F1 ciclo paráfrasis,0.028788,-0.04642,0.100983,False


#### Precisión del regex v1 en el gold

El regex marca pocas oraciones del gold, así que su precisión se estima con muy pocos casos. Se dan las coincidencias por etapa y el IC de Wilson (sin pesos, sobre los conteos).

In [7]:
from scipy.stats import norm
def wilson(k, n, a=0.05):
    if n == 0:
        return (np.nan, np.nan)
    z = norm.ppf(1 - a / 2); p_ = k / n
    c = (p_ + z**2 / (2 * n)) / (1 + z**2 / n); h = z * np.sqrt(p_ * (1 - p_) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return (c - h, c + h)

filas = []
Rx = P["Regex v1"] >= 0.5
rec_rx = m.metricas(Y, P["Regex v1"], E, pesos=W)[0].recall
for j, e in enumerate(E):
    n_, k_ = int(Rx[:, j].sum()), int((Rx[:, j] & (Y[:, j] == 1)).sum())
    lo, hi = wilson(k_, n_)
    filas.append({"etapa": e, "coincidencias del regex": n_, "aciertos": k_, "precisión": k_ / n_ if n_ else np.nan,
                  "IC Wilson inf": lo, "IC Wilson sup": hi, "recall (ponderado)": rec_rx[e]})
prec_regex = pd.DataFrame(filas).set_index("etapa")
print(f"El regex v1 marca {int(Rx.any(axis=1).sum())} de {len(gold)} oraciones del gold.")
prec_regex.round(3)

El regex v1 marca 29 de 400 oraciones del gold.


,coincidencias del regex,aciertos,precisión,IC Wilson inf,IC Wilson sup,recall (ponderado)
etapa,,,,,,
llegada,2,0,0.000,0.000,0.658,0.000
atencion,6,6,1.000,0.610,1.000,0.080
cocina_mesa,8,2,0.250,0.071,0.591,0.102
cobro,2,1,0.500,0.095,0.905,0.024
sabor,6,6,1.000,0.610,1.000,0.188
precio,7,4,0.571,0.250,0.842,0.315
local,2,2,1.000,0.342,1.000,0.064


In [8]:
fig, ax = plt.subplots(figsize=(8, 4.2))
orden = escalera.index[::-1]
ax.barh(orden, escalera.loc[orden, "F1 ciclo"], color="#9db7d5", label="gold completo")
ax.barh(orden, escalera.loc[orden, "F1 ciclo paráfrasis"], color="#b5483a", height=0.4, label="solo paráfrasis")
ax.axvline(0.70, color="black", ls="--", lw=1)
ax.set(xlabel=f"F1 macro del ciclo ({', '.join(MACRO)}), ponderado", title="La escalera en el gold de Barcelona")
ax.legend(loc="lower right", fontsize=8); plt.tight_layout(); plt.savefig(FIGURAS / "06_escalera_gold.png", dpi=150); plt.show()

<Figure size 800x420 with 1 Axes>

### 4. Umbrales y modelo elegido

In [9]:
kappa_ok = h04["kappa_ciclo_medio"] is not None and h04["kappa_ciclo_medio"] >= 0.65
umbrales = pd.DataFrame({
    m_: {"F1 ciclo ≥ 0.70": escalera.loc[m_, "F1 ciclo"] >= 0.70,
         "precisión atención ≥ 0.80": escalera.loc[m_, "precisión atención"] >= 0.80,
         "recall paráfrasis ≥ regex + 0.10": escalera.loc[m_, "recall paráfrasis"] >= f_regex_par + 0.10}
    for m_ in ORDEN_SIMPLE}).T
umbrales["LSTM > LogReg en paráfrasis"] = escalera.loc["LSTM", "F1 ciclo paráfrasis"] > escalera.loc["LogReg", "F1 ciclo paráfrasis"]
display(umbrales)
print(f"κ entre anotadores (ciclo): {h04['kappa_ciclo_medio']:.3f}. Mide consistencia entre anotadores, no validez.")

FINAL, RAZON = elegir(escalera)                       # regla de selección
apto = lambda n_: escalera.loc[n_, "F1 ciclo"] >= 0.70 and escalera.loc[n_, "precisión atención"] >= 0.80
aprendidos = ["Naive Bayes", "LogReg", "MLP", "RNN", "LSTM"]
aptos = [n_ for n_ in aprendidos if apto(n_)]
if aptos:
    PRODUCTO, RAZON_PRODUCTO = FINAL if FINAL in aptos else aptos[0], "modelo aprendido apto"
else:
    PRODUCTO = next(n_ for n_ in ["Regex v1", "Regex v2"] if escalera.loc[n_, "precisión atención"] >= 0.80)
    RAZON_PRODUCTO = ("La regla eligió LogReg; los umbrales, fijados antes, dicen que ningún modelo es apto para un gerente. "
                      "El producto vuelve al baseline que sí cumple precisión.") if FINAL == "LogReg" else "ningún modelo aprendido es apto"
print(f"\nSelección: {FINAL} ({RAZON}). Preferido en el dev de Madrid: {MODELO_DEV} (el modelo neuronal evaluado).")
print(f"Aptitud: modelos aprendidos aptos = {aptos or 'ninguno'} → el producto usa {PRODUCTO}.")
print(RAZON_PRODUCTO)

,F1 ciclo ≥ 0.70,precisión atención ≥ 0.80,recall paráfrasis ≥ regex + 0.10,LSTM > LogReg en paráfrasis
Regex v1,False,True,False,False
Regex v2,False,True,False,False
Naive Bayes,False,False,True,False
LogReg,False,False,True,False
MLP,False,False,True,False
RNN,False,False,True,False
LSTM,False,False,True,False


κ entre anotadores (ciclo): 0.949. Mide consistencia entre anotadores, no validez.

Selección: LogReg (ningún modelo cumple todos los umbrales; se usa LogReg y se reporta). Preferido en el dev de Madrid: LSTM (el modelo neuronal evaluado).
Aptitud: modelos aprendidos aptos = ninguno → el producto usa Regex v1.
La regla eligió LogReg; los umbrales, fijados antes, dicen que ningún modelo es apto para un gerente. El producto vuelve al baseline que sí cumple precisión.


### 5. Cobertura del ciclo (descriptiva, no es umbral)

Es el porcentaje de reseñas de 1-2★ de **Barcelona** con al menos una falla del ciclo, según cada detector, junto con su precisión ponderada en el gold. El umbral original (más de 54.5%) dejó de ser criterio porque el léxico v2 lo supera sin ningún modelo (03).

In [10]:
b = datos.cargar_oraciones("Barcelona", con_resena=True)
b = b[b.estrellas <= 2].reset_index(drop=True)
pred_b = {"Regex v1": et.etiquetas_regex(b.oracion, lexico.ETAPAS).values,
          "Regex v2": et.etiquetas_regex(b.oracion, lexico.ETAPAS_V2, guardia_negacion=True).values,
          "LogReg": (lin["logreg"].predict_proba(lin["vec"].transform(b.oracion)) >= 0.5).astype(int)}
red, vocab = m.cargar_red(clave_red("LSTM")); pred_b["LSTM"] = (m.proba_red(red, vocab, b.oracion) >= 0.5).astype(int)
ci = [E.index(e) for e in CICLO]
filas = []
for k, Yp in pred_b.items():
    por_res = pd.DataFrame(Yp, columns=E).groupby(b.review_id.values).max()
    tab, _, _ = m.metricas(Y, P[k], E, pesos=W)
    filas.append({"detector": k, "cobertura ciclo por reseña %": por_res[CICLO].any(axis=1).mean() * 100,
                  "cobertura 7 etapas por reseña %": por_res.any(axis=1).mean() * 100,
                  "cobertura ciclo por oración %": Yp[:, ci].any(axis=1).mean() * 100,
                  "precisión media del ciclo (gold)": tab.loc[MACRO, "precisión"].mean()})
cobertura = pd.DataFrame(filas).set_index("detector")
print(f"Barcelona, reseñas 1-2★: {b.review_id.nunique():,} ({len(b):,} oraciones)")
cobertura.round(2)

Barcelona, reseñas 1-2★: 41,456 (271,058 oraciones)


,cobertura ciclo por reseña %,cobertura 7 etapas por reseña %,cobertura ciclo por oración %,precisión media del ciclo (gold)
detector,,,,
Regex v1,27.37,50.08,5.14,0.59
Regex v2,34.82,60.30,6.86,0.59
LogReg,84.28,97.89,37.96,0.42
LSTM,77.14,97.25,31.39,0.38


### 6. Análisis de errores del modelo final

In [11]:
Pf = P[FINAL] >= 0.5
err_fn = (Y == 1) & ~Pf
err_fp = (Y == 0) & Pf
NEG = gold.oracion.str.lower().str.contains(r"\b(?:not|never|no|n't|nothing|nobody|wasn't|didn't)\b").values
MULTI = (Y.sum(axis=1) > 1)
cat = pd.DataFrame({
    "con negación": [NEG.mean(), err_fn.any(axis=1)[NEG].mean(), err_fp.any(axis=1)[NEG].mean()],
    "sin negación": [(~NEG).mean(), err_fn.any(axis=1)[~NEG].mean(), err_fp.any(axis=1)[~NEG].mean()],
    "varias etapas (gold)": [MULTI.mean(), err_fn.any(axis=1)[MULTI].mean(), err_fp.any(axis=1)[MULTI].mean()],
    "marcada dudosa": [gold.dudosa.mean(), err_fn.any(axis=1)[gold.dudosa.values == 1].mean() if gold.dudosa.sum() else np.nan,
                       err_fp.any(axis=1)[gold.dudosa.values == 1].mean() if gold.dudosa.sum() else np.nan],
}, index=["% del gold", "tasa de falsos negativos", "tasa de falsos positivos"]).T
display((cat * 100).round(1))

def muestra(mascara, n=5, semilla=0):
    idx = np.flatnonzero(mascara)
    return gold.iloc[np.random.default_rng(semilla).choice(idx, min(n, len(idx)), replace=False)]
print(f"Falsos negativos del ciclo ({FINAL}): {err_fn[:, ci].any(axis=1).sum()} | falsos positivos: {err_fp[:, ci].any(axis=1).sum()}")
ej_fn = muestra(err_fn[:, ci].any(axis=1)); ej_fp = muestra(err_fp[:, ci].any(axis=1), semilla=1)
for nombre, ej, M_ in [("NO detectó (gold dice falla)", ej_fn, err_fn), ("Marcó falla que el gold no tiene", ej_fp, err_fp)]:
    print(f"\n{nombre}:")
    for i, r in ej.iterrows():
        etq = [E[j] for j in range(len(E)) if M_[i, j]]
        print(f"  [{', '.join(etq)}] {r.oracion}")

,% del gold,tasa de falsos negativos,tasa de falsos positivos
con negación,18.2,34.2,47.9
sin negación,81.8,21.7,33.0
varias etapas (gold),9.0,80.6,27.8
marcada dudosa,8.5,64.7,44.1


Falsos negativos del ciclo (LogReg): 61 | falsos positivos: 87

NO detectó (gold dice falla):
  [atencion] Only one person at the check out, slow service at the counter, despite being lunch time.
  [llegada, atencion] Disappointing and hostile.
  [cobro] At first I was shocked; this was clearly an oversight and not a deliberate attempt to scrap 15c of the bill, but to come after us as we left the restaurant for such a small amount, I thought was extremely poor.
  [cobro, sabor] Breads were not fresh, staff were not welcoming, the receipt was not right, you should check it.
  [cobro] He wrote one out - well, what do you know, it was only 15.50 € after all.

Marcó falla que el gold no tiene:
  [cocina_mesa] The staff said no one else had a problem with the food, again this highlights their lack of customer service but also the fact the clientele frequenting this chain haven't had a decent meal before!
  [cobro] However, I insisted to have a bill, so he said that other waiter will print i

### 7. Sensibilidad y especificidad para el scorecard (Rogan-Gladen)

In [12]:
Yh = (P[FINAL] >= 0.5)
medicion = {}
for j, e in enumerate(E):
    pos, neg = Y[:, j] == 1, Y[:, j] == 0
    sens = np.average(Yh[pos, j], weights=W[pos]) if pos.any() else np.nan
    esp = np.average(~Yh[neg, j], weights=W[neg])
    medicion[e] = (sens, esp)
pd.DataFrame(medicion, index=["sensibilidad (recall)", "especificidad"]).T.round(3)

,sensibilidad (recall),especificidad
llegada,0.667,0.984
atencion,0.595,0.953
cocina_mesa,0.295,0.973
cobro,0.451,0.998
sabor,0.548,0.898
precio,0.764,0.927
local,0.408,0.973


### 8. Fila post-hoc / exploratoria

**Criterio, escrito antes de correr esta celda:** umbral por etapa = el menor que da precisión ≥ 0.80 en el dev de paráfrasis reales de Madrid (05b, predicciones fuera de partición); si ninguno la alcanza, se usa el que maximiza F1. Se evalúa **una sola vez** en el gold y **no cambia el scorecard**. Después, el gold ya no se toca.

**Advertencia:** el dev de Madrid son solo oraciones de reseñas 1-2★ sin léxico, y ese mismo dev ordenó mal a los modelos: prefirió el LSTM, que en el gold no se distingue de LogReg.

In [13]:
h05b = resultados.cargar("05b")
U = h05b["umbrales_posthoc"]
P_post = (P["LogReg"] >= np.array([U[e] for e in E])).astype(float)
posthoc = pd.DataFrame([fila("LogReg post-hoc (umbral calibrado en Madrid)", P_post)]).set_index("modelo")
print("Umbrales:", U, "| reglas:", h05b["regla_posthoc"])
display(pd.concat([escalera.loc[["LogReg"]], posthoc])[["F1 ciclo", "precisión atención", "recall atención", "recall paráfrasis", "F1 ciclo paráfrasis"]].round(3))
print("¿Apta?", bool(posthoc.iloc[0]["F1 ciclo"] >= 0.70 and posthoc.iloc[0]["precisión atención"] >= 0.80), "— exploratoria: no cambia el producto")

Umbrales: {'atencion': 0.95, 'cobro': 0.15, 'cocina_mesa': 0.7, 'llegada': 0.3, 'local': 0.8, 'precio': 0.9, 'sabor': 0.95} | reglas: {'atencion': 'precisión ≥ 0.80', 'cobro': 'máximo F1 (ninguno llega a 0.80)', 'cocina_mesa': 'máximo F1 (ninguno llega a 0.80)', 'llegada': 'máximo F1 (ninguno llega a 0.80)', 'local': 'precisión ≥ 0.80', 'precio': 'precisión ≥ 0.80', 'sabor': 'precisión ≥ 0.80'}


,F1 ciclo,precisión atención,recall atención,recall paráfrasis,F1 ciclo paráfrasis
modelo,,,,,
LogReg,0.417,0.422,0.595,0.518,0.397
LogReg post-hoc (umbral calibrado en Madrid),0.281,0.682,0.185,0.221,0.236


¿Apta? False — exploratoria: no cambia el producto


### Lectura

**Selección y aptitud.** Hay dos reglas previas, y se aplican las dos. La de selección elige LogReg: el LSTM, preferido en el dev de Madrid, no le gana en las paráfrasis. La de aptitud dice que **ningún modelo aprendido sirve para un gerente**: el mejor F1 macro del ciclo es 0.417 (LogReg, IC 95% de 0.314 a 0.523), contra el 0.70 pedido, y la precisión de Atención es 0.422, contra 0.80. **La regla eligió LogReg; los umbrales, fijados antes, dicen que ningún modelo es apto para un gerente. El producto vuelve al baseline que sí cumple precisión.** Ese baseline es el regex v1.

**LogReg contra el LSTM (el modelo neuronal evaluado): no son distinguibles.** En el bootstrap pareado, la diferencia de F1 del ciclo es +0.028 (IC de −0.046 a 0.098) y en las paráfrasis +0.029 (IC de −0.046 a 0.101). Los dos intervalos incluyen el 0. El dev de Madrid había preferido el LSTM, así que ese dev, de solo 300 oraciones de 1-2★, ordenó mal a los modelos.

**El regex v1 es preciso, pero con muy pocos casos.** Marca 29 de las 400 oraciones del gold. En Atención acierta 6 de 6 (IC de Wilson de 0.61 a 1.00) y en Sabor también 6 de 6; en Cocina → mesa solo 2 de 8. Su recall ponderado es bajo en todas las etapas (0.08 en Atención): ve una fracción chica de las quejas. Por eso el scorecard rotula sus tasas como **quejas explícitas (piso de la tasa real)**.

**Lo que muestra el subconjunto de paráfrasis** (371 oraciones donde el v1 no marca nada; el entrenamiento usa el v2, que ahí recupera 0.027): los modelos aprendidos sí ven lo que el léxico no ve (recall de 0.518 con LogReg), pero a costa de la precisión. LogReg marca una falla del ciclo en el 84.3% de las reseñas 1-2★ de Barcelona, contra el 27.4% del regex, y su error no es parejo: se dispara con cualquier tono negativo (05b: también marca el 27.5% de las reseñas de 5★).

**Fila post-hoc / exploratoria** (umbral por etapa calibrado solo en el dev de Madrid y evaluado una vez): la precisión de Atención sube a 0.682, pero el F1 del ciclo baja a 0.281 y el recall en paráfrasis a 0.221. Tampoco es apta, y no cambia el producto.

**Errores** (LogReg con umbral 0.5): los falsos negativos suben en oraciones con negación (34% contra 22%), con varias etapas (81%) y en las que el anotador marcó dudosas (65%). La sección 6 lista ejemplos reales de los dos tipos de error.

**Llegada** tiene 6 positivos en el gold (0.25% de Barcelona, ponderado) y queda fuera del macro: el gold no alcanza para medirla. Aunque se sobre-muestreó el tema, casi ninguna mención de mesa o reserva es una falla.

**κ entre anotadores = 0.949** en el ciclo: mide consistencia, no validez.

### Cifras clave

In [14]:
ej_par = gold[gold.parafrasis & gold[MACRO].any(axis=1)].sample(3, random_state=0).oracion.tolist()
resultados.guardar("06", {
    "n_gold": len(gold), "n_parafrasis": int(gold.parafrasis.sum()), "macro": MACRO, "variante": VARIANTE,
    "escalera": escalera, "ic": {k: v for k, v in ic.items()}, "umbrales": umbrales, "kappa_ok": kappa_ok,
    "final": FINAL, "razon_final": RAZON, "modelo_preferido_dev": MODELO_DEV, "cobertura": cobertura,
    "producto": PRODUCTO, "razon_producto": RAZON_PRODUCTO, "aptos": aptos, "diferencia_logreg_lstm": dif,
    "precision_regex": prec_regex, "posthoc": posthoc.iloc[0], "umbrales_posthoc": U, "kappa_ciclo": h04["kappa_ciclo_medio"],
    "positivos_ponderados_pct": {e: float(np.average(gold[e], weights=W) * 100) for e in E},
    "positivos_parafrasis": gold.loc[gold.parafrasis, E].sum(),
    "errores_categorias": cat, "ejemplos_fn": ej_fn.oracion.tolist(), "ejemplos_fp": ej_fp.oracion.tolist(),
    "ejemplos_parafrasis_gold": ej_par, "medicion": {e: list(v) for e, v in medicion.items()},
    "positivos_gold": gold[E].sum(),
})

hallazgos_06.json: 28 claves → aptos, cobertura, diferencia_logreg_lstm, ejemplos_fn, ejemplos_fp, ejemplos_parafrasis_gold, errores_categorias, escalera, final, ic, kappa_ciclo, kappa_ok, macro, medicion, modelo_preferido_dev, n_gold, n_parafrasis, positivos_gold, positivos_parafrasis, positivos_ponderados_pct, posthoc, precision_regex, producto, razon_final, razon_producto, umbrales, umbrales_posthoc, variante


PosixPath('/Users/georgealbadr/Github/Natural_Language_Processing/resultados/hallazgos_06.json')

### Hallazgos

- **Ningún modelo aprendido es apto** en el gold de Barcelona (400 oraciones, evaluado una vez, métricas ponderadas). El mejor F1 macro del ciclo es 0.417 (LogReg) contra 0.70, y la precisión de Atención 0.422 contra 0.80. **La regla eligió LogReg; los umbrales, fijados antes, dicen que ningún modelo es apto para un gerente. El producto vuelve al baseline que sí cumple precisión** (regex v1).
- **LogReg y el LSTM no se distinguen:** diferencia pareada +0.028, IC de −0.046 a 0.098.
- **El regex v1 es preciso pero ve poco:** acierta 6 de 6 en Atención (IC de Wilson de 0.61 a 1.00), con recall de 0.08. Sus tasas son un piso de la tasa real.
- **Los modelos ven paráfrasis (recall 0.518) pero marcan de más:** el 84.3% de las reseñas 1-2★ y el 27.5% de las de 5★. El error es diferencial: sigue al tono negativo.
- **Fila post-hoc (exploratoria):** con el umbral calibrado en Madrid, la precisión de Atención sube a 0.682 y el F1 baja a 0.281. Tampoco es apta.

### Verificación de citas

Toda oración citada entre comillas en este notebook tiene que existir literalmente en el corpus.

In [15]:
citas = []
verificadas = datos.verificar_citas(citas, ('Barcelona',))
print(f"{len(verificadas)} citas verificadas:", verificadas)

0 citas verificadas: {}
